# LeetCode 472 — Concatenated Words (Hard) bằng Python

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def findAllConcatenatedWordsInADict(self, words: List[str]) -> List[str]:

```

**Mục tiêu sau khi học xong:**

1. Hiểu chính xác thế nào là một **"từ ghép" (concatenated word)**: ghép **trọn vẹn** từ **ít nhất 2** từ **ngắn hơn** trong chính danh sách `words`.
2. Ôn lại **Word Break (LeetCode 139)** bằng **DP (quy hoạch động)** từ căn bản — vì bài 472 thực chất là "chạy Word Break cho từng từ".
3. Viết **Cách 1 — brute force đệ quy**: thử mọi cách cắt một từ, và thấy vì sao nó nổ **hàm mũ**.
4. Viết **Cách 2 — Word Break DP cho từng từ** (cách chính để nộp), hiểu hai chi tiết sống còn: **loại chính từ đó ra** và **bỏ qua chuỗi rỗng**.
5. Biết **Cách 3 — sắp theo độ dài**, chỉ dùng các từ ngắn hơn đã thêm vào `set`.
6. Biết **Cách 4 — DFS + memo** (lời giải 5 dòng rất hay gặp trong phỏng vấn) và **Cách 5 — Trie** (cây tiền tố, tùy chọn) để bỏ chi phí cắt chuỗi.
7. Phân tích độ phức tạp `O(N·L³)` so với `O(N·L²)`, và biết vì sao chênh lệch đó đến từ **việc cắt chuỗi** chứ không phải từ vòng lặp.

## 1. Đề bài

Cho một mảng chuỗi `words` (**không có phần tử trùng nhau**), hãy trả về **tất cả các từ ghép** (concatenated words) có trong danh sách.

Một **từ ghép** là một chuỗi được tạo thành **hoàn toàn** từ **ít nhất hai từ ngắn hơn** (không nhất thiết khác nhau) có trong mảng đã cho.

**Ví dụ 1**

```text
Input:  words = ["cat","cats","catsdogcats","dog","dogcatsdog","hippopotamuses","rat","ratcatdogcat"]
Output: ["catsdogcats","dogcatsdog","ratcatdogcat"]
Explanation: "catsdogcats" can be concatenated by "cats", "dog" and "cats";
"dogcatsdog" can be concatenated by "dog", "cats" and "dog";
"ratcatdogcat" can be concatenated by "rat", "cat", "dog" and "cat".
```

**Ví dụ 2**

```text
Input:  words = ["cat","dog","catdog"]
Output: ["catdog"]
```

**Constraints (ràng buộc):**

- `1 <= words.length <= 10^4`
- `1 <= words[i].length <= 30`
- `words[i]` chỉ gồm chữ cái tiếng Anh viết thường.
- Mọi chuỗi trong `words` là **khác nhau** (unique).
- `1 <= sum(words[i].length) <= 10^5`

**Ghi chú về chuỗi rỗng:** một số phiên bản cũ của đề cho phép `words[i].length = 0` (tức có thể có chuỗi rỗng `""` trong input). Đề hiện tại không còn trường hợp này, nhưng code tốt nên **phòng thủ**: chuỗi rỗng **không bao giờ** là từ ghép, và cũng **không được** dùng làm "mảnh ghép". Ta sẽ thấy ở mục 20 vì sao quên chi tiết này làm code sai.

### Đề nói "trả về tất cả" — thứ tự thế nào?

LeetCode chấp nhận kết quả theo **thứ tự bất kỳ**. Vì vậy khi tự test, ta so sánh **sau khi sắp xếp** bằng `sorted(...)`. Cell dưới lưu 2 ví dụ của đề và hàm nhỏ `same_answer` dùng xuyên suốt notebook.

In [1]:
from typing import List

examples = [
    (["cat", "cats", "catsdogcats", "dog", "dogcatsdog", "hippopotamuses", "rat", "ratcatdogcat"],
     ["catsdogcats", "dogcatsdog", "ratcatdogcat"]),
    (["cat", "dog", "catdog"],
     ["catdog"]),
]

for words, expected in examples:
    print("words    =", words)
    print("expected =", expected)
    print()


def same_answer(actual: List[str], expected: List[str]) -> bool:
    # So sánh 2 list từ, bỏ qua thứ tự
    return sorted(actual) == sorted(expected)

print(["a", "b"] == ["b", "a"])              # False: list so sánh cả thứ tự
print(same_answer(["a", "b"], ["b", "a"]))   # True

words    = ['cat', 'cats', 'catsdogcats', 'dog', 'dogcatsdog', 'hippopotamuses', 'rat', 'ratcatdogcat']
expected = ['catsdogcats', 'dogcatsdog', 'ratcatdogcat']

words    = ['cat', 'dog', 'catdog']
expected = ['catdog']

False
True


### Hiểu định nghĩa bằng tay trước khi code

Ta tự kiểm tra từng từ của Ví dụ 1. Câu hỏi cho mỗi từ `w`: **có cách nào cắt `w` thành ≥ 2 mảnh, mà mỗi mảnh là một từ trong `words`?**

| Từ `w` | Một cách cắt thử | Kết luận |
|---|---|---|
| `"cat"` | Chỉ có thể là chính nó (1 mảnh) — `"c"`, `"ca"`, `"at"`... không phải từ | ✘ không ghép |
| `"cats"` | `"cat" + "s"` — nhưng `"s"` không phải từ | ✘ |
| `"catsdogcats"` | `"cats" + "dog" + "cats"` | ✔ 3 mảnh |
| `"dogcatsdog"` | `"dog" + "cats" + "dog"` | ✔ 3 mảnh |
| `"hippopotamuses"` | Không có tiền tố nào là từ | ✘ |
| `"ratcatdogcat"` | `"rat" + "cat" + "dog" + "cat"` | ✔ 4 mảnh |

Ba điều rút ra từ bảng:

1. **Ít nhất 2 mảnh**: một từ "là chính nó" (1 mảnh) **không** được tính. Nếu không, mọi từ đều là từ ghép!
2. **Mảnh dùng lại được**: `"catsdogcats"` dùng `"cats"` hai lần — đề nói "không nhất thiết khác nhau".
3. **Phải phủ kín**: mọi ký tự của `w` phải thuộc một mảnh nào đó; không được thừa ký tự (`"cats"` thừa `"s"`).

Cell dưới kiểm chứng "bằng tay" các cách cắt ở bảng: ghép lại các mảnh phải ra đúng `w`, và mỗi mảnh phải có trong danh sách.

In [2]:
words = examples[0][0]
word_set = set(words)

for w, pieces in [("catsdogcats", ["cats", "dog", "cats"]),
                  ("dogcatsdog", ["dog", "cats", "dog"]),
                  ("ratcatdogcat", ["rat", "cat", "dog", "cat"]),
                  ("cats", ["cat", "s"])]:
    joined_ok = "".join(pieces) == w                  # ghép lại có ra đúng w?
    all_words = all(p in word_set for p in pieces)    # mọi mảnh đều là từ?
    enough = len(pieces) >= 2                         # ít nhất 2 mảnh?
    print(f"{w:<13} = {' + '.join(pieces):<24} | ghép đúng={joined_ok}, mọi mảnh là từ={all_words}, >=2 mảnh={enough}")

catsdogcats   = cats + dog + cats        | ghép đúng=True, mọi mảnh là từ=True, >=2 mảnh=True
dogcatsdog    = dog + cats + dog         | ghép đúng=True, mọi mảnh là từ=True, >=2 mảnh=True
ratcatdogcat  = rat + cat + dog + cat    | ghép đúng=True, mọi mảnh là từ=True, >=2 mảnh=True
cats          = cat + s                  | ghép đúng=True, mọi mảnh là từ=False, >=2 mảnh=True


## 2. Kiến thức nền Python cho bài này

### 2.1. Cắt chuỗi (slicing): tiền tố, hậu tố, đoạn giữa

- `w[a:b]` lấy các ký tự từ chỉ số `a` **đến trước** `b` (không lấy `b`). Độ dài đoạn là `b - a`.
- **Tiền tố (prefix)**: đoạn đầu `w[:i]`. **Hậu tố (suffix)**: đoạn cuối `w[i:]`.
- Nếu `a == b` thì `w[a:b]` là **chuỗi rỗng** `""`.

Với `w = "catsdogcats"` (chỉ số `0..10`, độ dài 11):

| Biểu thức | Kết quả |
|---|---|
| `w[:3]` | `"cat"` |
| `w[:4]` | `"cats"` |
| `w[4:7]` | `"dog"` |
| `w[7:]` | `"cats"` |
| `w[5:5]` | `""` |

**Chi phí cần nhớ:** `w[a:b]` tạo ra một **chuỗi mới** — Python phải chép `b - a` ký tự, nên tốn `O(b - a)` thời gian. Chi tiết nhỏ này sẽ quyết định độ phức tạp `L²` hay `L³` ở mục 9.

In [3]:
w = "catsdogcats"
print(len(w))
print(w[:3], w[:4], w[4:7], w[7:])
print(repr(w[5:5]))               # chuỗi rỗng

# Liệt kê mọi cách cắt w thành (tiền tố, hậu tố) với cả hai đều KHÁC rỗng
for i in range(1, len(w)):
    if i in (3, 4, 7):            # chỉ in vài dòng đáng chú ý
        print(f"i={i}: {w[:i]!r} + {w[i:]!r}")

11
cat cats dog cats
''
i=3: 'cat' + 'sdogcats'
i=4: 'cats' + 'dogcats'
i=7: 'catsdog' + 'cats'


### 2.2. `set`: tra cứu nhanh, thêm và bớt phần tử

Ta sẽ hỏi câu "đoạn này có phải là một từ trong danh sách không?" **rất nhiều lần**.

- Với `list`, `x in my_list` phải dò **từng phần tử**: tốn `O(N)` phép so sánh (N có thể tới `10^4`).
- Với `set` (tập hợp, cài bằng **bảng băm — hash table**), `x in my_set` chỉ tốn **trung bình** `O(len(x))` để băm chuỗi, **không phụ thuộc** số phần tử.

Ba thao tác `set` dùng trong bài:

| Thao tác | Ý nghĩa |
|---|---|
| `s.add(x)` | thêm `x` |
| `s.discard(x)` | bỏ `x` nếu có (không báo lỗi nếu không có; khác với `s.remove(x)` sẽ báo `KeyError`) |
| `x in s` | kiểm tra `x` có trong `s` không |

Thủ thuật "**tạm bỏ rồi thêm lại**" (`discard` → kiểm tra → `add`) giúp ta hỏi: "từ `w` có ghép được từ **các từ khác** `w` không?" mà không cần tạo `set` mới cho mỗi từ.

In [4]:
words = ["cat", "cats", "dog", "catdog"]
word_set = set(words)
print(sorted(word_set))           # set không có thứ tự -> in bản đã sắp xếp cho dễ nhìn
print("cat" in word_set)          # True
print("ca" in word_set)           # False

word_set.discard("catdog")        # tạm bỏ "catdog"
print("sau discard:", sorted(word_set), "| 'catdog' in set?", "catdog" in word_set)
word_set.discard("xyz")           # bỏ phần tử không có -> không lỗi
word_set.add("catdog")            # thêm lại
print("sau add    :", sorted(word_set))

try:
    word_set.remove("xyz")        # remove phần tử không có -> KeyError
except KeyError as e:
    print("remove báo lỗi KeyError:", e)

['cat', 'catdog', 'cats', 'dog']
True
False
sau discard: ['cat', 'cats', 'dog'] | 'catdog' in set? False
sau add    : ['cat', 'catdog', 'cats', 'dog']
remove báo lỗi KeyError: 'xyz'


### 2.3. Sắp xếp theo độ dài: `sorted(words, key=len)`

`sorted(seq, key=f)` sắp xếp theo giá trị `f(x)` của từng phần tử. Với `key=len`, từ ngắn đứng trước.

Sắp xếp của Python là **ổn định (stable)**: hai từ cùng độ dài giữ nguyên thứ tự ban đầu. Ta sẽ dùng nó ở Cách 3, với nhận xét: **mọi mảnh ghép của `w` đều ngắn hơn `w`**, nên nếu xử lý từ ngắn trước, lúc tới `w` thì mọi "nguyên liệu" có thể dùng đã sẵn sàng.

In [5]:
words = ["cat", "cats", "catsdogcats", "dog", "dogcatsdog", "hippopotamuses", "rat", "ratcatdogcat"]
for w in sorted(words, key=len):
    print(f"{len(w):>2}  {w}")

 3  cat
 3  dog
 3  rat
 4  cats
10  dogcatsdog
11  catsdogcats
12  ratcatdogcat
14  hippopotamuses


## 3. Ôn lại Word Break (LeetCode 139) bằng DP từ căn bản

**Word Break (139):** cho chuỗi `s` và từ điển `wordDict`, trả về `True` nếu `s` **có thể** cắt thành dãy các từ trong từ điển (từ dùng lại được), ngược lại `False`.

Vì sao học bài này trước? Vì câu hỏi của bài 472 cho **một** từ `w` chính là: "**Word Break của `w`**, với từ điển là **các từ khác** trong `words`". Nắm chắc Word Break là nắm 80% bài 472.

### DP là gì? (nói đơn giản)

**Quy hoạch động (Dynamic Programming — DP)** = chia bài lớn thành **bài con** nhỏ hơn, giải mỗi bài con **một lần**, lưu vào **bảng**, rồi dùng lại kết quả đã lưu để giải bài lớn hơn.

### Định nghĩa bảng `dp`

> `dp[i] = True` nếu **tiền tố** `s[:i]` (i ký tự đầu) cắt được thành các từ.

- **Điểm tựa:** `dp[0] = True` — tiền tố rỗng "cắt được" bằng 0 từ. Không có điểm tựa thì không ô nào bật được thành `True`.
- **Công thức chuyển:** `dp[i] = True` nếu tồn tại `j < i` sao cho `dp[j] = True` **và** `s[j:i]` là một từ.

Đọc bằng lời: "`s[:i]` cắt được nếu ta cắt được một đoạn đầu `s[:j]`, và **mảnh cuối** `s[j:i]` là một từ trọn vẹn."

### Tính tay với `s = "catsdog"`, `wordDict = ["cat","cats","dog"]`

Chỉ số: `c0 a1 t2 s3 d4 o5 g6`, độ dài 7, bảng `dp` có 8 ô `dp[0..7]`.

| i | `s[:i]` | `j` làm nó bật `True` | Mảnh cuối `s[j:i]` | `dp[i]` |
|---:|---|---:|---|:---:|
| 0 | `""` | — | (điểm tựa) | T |
| 1, 2 | `"c"`, `"ca"` | không có | | F |
| 3 | `"cat"` | 0 | `"cat"` | T |
| 4 | `"cats"` | 0 | `"cats"` | T |
| 5, 6 | `"catsd"`, `"catsdo"` | không có | | F |
| 7 | `"catsdog"` | 4 | `"dog"` | T |

`dp[7] = True` → `"catsdog"` cắt được: `"cats" + "dog"`. Để ý `dp[3] = True` (qua `"cat"`) nhưng không dẫn tới đâu, vì `s[3:7] = "sdog"` không phải từ.

In [6]:
def word_break_139(s: str, wordDict: List[str]) -> bool:
    word_set = set(wordDict)
    n = len(s)
    dp = [False] * (n + 1)
    dp[0] = True                        # tiền tố rỗng: cắt được bằng 0 từ
    for i in range(1, n + 1):           # i = độ dài tiền tố đang xét
        for j in range(i):              # j = chỗ bắt đầu của mảnh cuối
            if dp[j] and s[j:i] in word_set:
                dp[i] = True
                break                   # chỉ cần 1 cách là đủ
    return dp[n]

print(word_break_139("catsdog", ["cat", "cats", "dog"]))                   # True
print(word_break_139("leetcode", ["leet", "code"]))                        # True
print(word_break_139("applepenapple", ["apple", "pen"]))                   # True (dùng lại "apple")
print(word_break_139("catsandog", ["cats", "dog", "sand", "and", "cat"]))  # False

True
True
True
False


### Trace Word Break: nhìn bảng `dp` được bật từng ô

Hàm `trace_word_break_139` in ra, với mỗi `i` có `dp[i] = True`, cái `j` và mảnh `s[j:i]` đã "mở khóa" nó; cuối cùng in cả bảng dưới dạng chuỗi `T`/`.` cho dễ nhìn. So sánh output với bảng tính tay ở trên.

In [7]:
def trace_word_break_139(s: str, wordDict: List[str]) -> bool:
    word_set = set(wordDict)
    n = len(s)
    dp = [False] * (n + 1)
    dp[0] = True
    for i in range(1, n + 1):
        for j in range(i):
            if dp[j] and s[j:i] in word_set:
                dp[i] = True
                print(f"  dp[{i}] = True  vì dp[{j}] = True và s[{j}:{i}] = {s[j:i]!r} là từ")
                break
    print("  chỉ số :", "".join(str(i % 10) for i in range(n + 1)))
    print("  bảng dp:", "".join("T" if x else "." for x in dp), "-> kết quả", dp[n])
    return dp[n]

print("s = 'catsdog'")
trace_word_break_139("catsdog", ["cat", "cats", "dog"])
print("s = 'catsandog'")
trace_word_break_139("catsandog", ["cats", "dog", "sand", "and", "cat"])

s = 'catsdog'
  dp[3] = True  vì dp[0] = True và s[0:3] = 'cat' là từ
  dp[4] = True  vì dp[0] = True và s[0:4] = 'cats' là từ
  dp[7] = True  vì dp[4] = True và s[4:7] = 'dog' là từ
  chỉ số : 01234567
  bảng dp: T..TT..T -> kết quả True
s = 'catsandog'
  dp[3] = True  vì dp[0] = True và s[0:3] = 'cat' là từ
  dp[4] = True  vì dp[0] = True và s[0:4] = 'cats' là từ
  dp[7] = True  vì dp[3] = True và s[3:7] = 'sand' là từ
  chỉ số : 0123456789
  bảng dp: T..TT..T.. -> kết quả False


False

## 4. Nối Word Break với bài 472: "loại chính nó ra"

Ý tưởng tự nhiên nhất: với mỗi từ `w`, gọi `word_break_139(w, words)`. Nhưng thử ngay sẽ thấy **sai**: mọi từ đều ra `True`!

**Vì sao?** Vì `w` có trong từ điển, nên ở bước `i = len(w)`, `j = 0`: `dp[0] = True` và `w[0:len(w)] = w` là từ → `dp[len(w)] = True`. Tức là ta đã cắt `w` thành **1 mảnh là chính nó** — điều đề cấm.

**Cách sửa (hai lựa chọn tương đương):**

1. **Bỏ `w` khỏi từ điển** khi kiểm tra `w` (dùng `discard` rồi `add` lại).
2. **Giữ nguyên từ điển** nhưng **cấm** đúng một cặp `(j, i) = (0, len(w))` — tức cấm mảnh duy nhất phủ cả từ.

**Vì sao chỉ cần vậy là đủ "≥ 2 mảnh"?** Mọi mảnh `s[j:i]` với `j < i` đều **khác rỗng**. Nếu ta cấm mảnh dài bằng cả từ, thì mảnh nào cũng ngắn hơn `w` → cần ít nhất 2 mảnh mới phủ kín `w`. Và vì `words` không trùng nhau, "mảnh dài bằng `w` có trong `words`" chỉ có thể là chính `w`.

Cell dưới cho thấy lỗi và cách sửa trên Ví dụ 1.

In [8]:
words = examples[0][0]

print("Gọi thẳng Word Break với từ điển = mọi từ (SAI):")
print("  ", [w for w in words if word_break_139(w, words)])

print("Bỏ chính w ra khỏi từ điển (ĐÚNG):")
print("  ", [w for w in words if word_break_139(w, [x for x in words if x != w])])

Gọi thẳng Word Break với từ điển = mọi từ (SAI):
   ['cat', 'cats', 'catsdogcats', 'dog', 'dogcatsdog', 'hippopotamuses', 'rat', 'ratcatdogcat']
Bỏ chính w ra khỏi từ điển (ĐÚNG):
   ['catsdogcats', 'dogcatsdog', 'ratcatdogcat']


## 5. Kiến thức nền: đệ quy (recursion)

Trước khi có DP, cách người mới nghĩ ra đầu tiên thường là **thử mọi cách cắt**. Để "thử mọi cách" một cách gọn gàng, ta dùng **đệ quy**.

**Đệ quy** = một hàm **tự gọi lại chính nó** với bài **nhỏ hơn**, cho tới khi gặp **trường hợp cơ sở (base case)** đủ nhỏ để trả lời ngay.

Với bài cắt chuỗi, bài nhỏ hơn rất tự nhiên: "**chọn mảnh đầu tiên** `w[start:end]`, rồi cắt tiếp **phần còn lại** `w[end:]`". Base case: `start == len(w)` — đã cắt hết, không còn gì.

### Ví dụ mini: liệt kê mọi cách cắt chuỗi `"abcd"` thành các mảnh (mảnh nào cũng được)

Kỹ thuật dùng ở đây là **backtracking (quay lui)**: (1) chọn mảnh → `path.append`, (2) đệ quy cắt tiếp, (3) **hoàn tác** → `path.pop()` để thử mảnh khác.

In [9]:
def all_splits(s: str) -> List[List[str]]:
    result = []
    path = []

    def go(start: int) -> None:
        if start == len(s):              # base case: đã cắt hết chuỗi
            result.append(path.copy())   # copy! vì path còn bị sửa tiếp
            return
        for end in range(start + 1, len(s) + 1):
            path.append(s[start:end])    # 1. chọn mảnh đầu
            go(end)                      # 2. cắt tiếp phần còn lại
            path.pop()                   # 3. hoàn tác để thử mảnh khác

    go(0)
    return result

for parts in all_splits("abcd"):
    print(parts)
print("Số cách cắt:", len(all_splits("abcd")))

['a', 'b', 'c', 'd']
['a', 'b', 'cd']
['a', 'bc', 'd']
['a', 'bcd']
['ab', 'c', 'd']
['ab', 'cd']
['abc', 'd']
['abcd']
Số cách cắt: 8


### Có bao nhiêu cách cắt một từ dài `L`?

Giữa `L` ký tự có `L - 1` "khe hở". Mỗi khe ta **cắt** hoặc **không cắt** → `2^(L-1)` cách. Với `"abcd"`: `2^3 = 8` — đúng như output. Với `L = 30` (giới hạn đề): `2^29 ≈ 5.4 × 10^8` cách cho **một** từ — quá nhiều.

## 6. Cách 1 — Brute force đệ quy: thử cắt, chỉ đi tiếp khi mảnh là từ

Ta không cần liệt kê hết cách cắt rồi mới lọc. Khôn hơn một chút: tại vị trí `start`, chỉ đi tiếp vào `end` nếu mảnh `w[start:end]` **là một từ**. Và vì bài chỉ hỏi **có hay không**, gặp một cách thành công là **trả `True` ngay** (không cần tìm tiếp).

Để bảo đảm "≥ 2 mảnh", ta mang theo biến `count` = số mảnh đã chọn:

```python
def can_split(start, count):
    if start == L:
        return count >= 2            # cắt hết: chỉ hợp lệ khi >= 2 mảnh
    for end in range(start + 1, L + 1):
        if w[start:end] in word_set and can_split(end, count + 1):
            return True
    return False
```

Ở đây `word_set` chứa **mọi** từ (kể cả `w`). Nếu ta chọn mảnh đầu là cả `w` thì tới `start == L` với `count = 1` → `False`. Điều kiện `count >= 2` tự loại trường hợp "1 mảnh là chính nó".

Chuỗi rỗng `""` cũng được xử lý đúng: `L = 0` nên ngay lập tức `start == L` với `count = 0` → `False`.

In [10]:
def concat_brute(words: List[str]) -> List[str]:
    word_set = set(words)
    result = []
    for w in words:
        L = len(w)

        def can_split(start: int, count: int) -> bool:
            if start == L:
                return count >= 2
            for end in range(start + 1, L + 1):
                if w[start:end] in word_set and can_split(end, count + 1):
                    return True
            return False

        if can_split(0, 0):
            result.append(w)
    return result

for words, expected in examples:
    got = concat_brute(words)
    print(got)
    assert same_answer(got, expected)
print("Cách 1 PASS trên 2 ví dụ đề")

['catsdogcats', 'dogcatsdog', 'ratcatdogcat']
['catdog']
Cách 1 PASS trên 2 ví dụ đề


### Trace Cách 1 trên `"catsdogcats"`: cây thử–sai

Hàm `trace_brute` in mỗi lần **chọn một mảnh là từ** (thụt lề theo độ sâu đệ quy), mỗi khi **cắt hết** (✔ hoặc ✘ tùy `count`), và mỗi khi đứng ở vị trí mà **không mảnh nào** đi tiếp được (✘ ngõ cụt).

Dự đoán trước khi chạy: thử mảnh đầu `"cat"` → còn `"sdogcats"`, không có từ nào bắt đầu bằng `"s"` → ngõ cụt; lùi lại thử `"cats"` → `"dog"` → `"cat"` (còn `"s"`, ngõ cụt) → lùi lại thử `"cats"` → hết chuỗi với 3 mảnh ✔.

In [11]:
def trace_brute(w: str, words: List[str]) -> bool:
    word_set = set(words)
    L = len(w)

    def can_split(start: int, count: int, depth: int) -> bool:
        pad = "    " * depth
        if start == L:
            ok = count >= 2
            print(f"{pad}hết chuỗi với {count} mảnh -> {'✔' if ok else '✘ (chưa đủ 2 mảnh)'}")
            return ok
        moved = False
        for end in range(start + 1, L + 1):
            piece = w[start:end]
            if piece in word_set:
                moved = True
                print(f"{pad}start={start}: thử mảnh {piece!r}")
                if can_split(end, count + 1, depth + 1):
                    return True
        if not moved:
            print(f"{pad}start={start}: phần còn lại {w[start:]!r} không bắt đầu bằng từ nào ✘")
        return False

    ans = can_split(0, 0, 0)
    print("Kết quả:", ans)
    return ans

words1 = examples[0][0]
trace_brute("catsdogcats", words1)
print("-" * 40)
trace_brute("cats", words1)

start=0: thử mảnh 'cat'
    start=3: phần còn lại 'sdogcats' không bắt đầu bằng từ nào ✘
start=0: thử mảnh 'cats'
    start=4: thử mảnh 'dog'
        start=7: thử mảnh 'cat'
            start=10: phần còn lại 's' không bắt đầu bằng từ nào ✘
        start=7: thử mảnh 'cats'
            hết chuỗi với 3 mảnh -> ✔
Kết quả: True
----------------------------------------
start=0: thử mảnh 'cat'
    start=3: phần còn lại 's' không bắt đầu bằng từ nào ✘
start=0: thử mảnh 'cats'
    hết chuỗi với 1 mảnh -> ✘ (chưa đủ 2 mảnh)
Kết quả: False


False

### Đọc trace bằng bảng

**Trace `"catsdogcats"`:**

| Bước | `start` | Mảnh thử | Chuyện gì xảy ra |
|---:|---:|---|---|
| 1 | 0 | `"cat"` | Đi tới `start=3`, còn `"sdogcats"` — không từ nào bắt đầu bằng `"s"` → ✘, lùi lại |
| 2 | 0 | `"cats"` | Đi tới `start=4` |
| 3 | 4 | `"dog"` | Đi tới `start=7` |
| 4 | 7 | `"cat"` | Đi tới `start=10`, còn `"s"` → ✘, lùi lại |
| 5 | 7 | `"cats"` | Tới `start=11 = L` với 3 mảnh → ✔, trả `True` ngay |

**Trace `"cats"`:** thử `"cat"` → còn `"s"` ✘; thử `"cats"` (cả từ) → hết chuỗi nhưng chỉ **1 mảnh** → ✘. Đây chính là chỗ `count >= 2` cứu ta khỏi việc nhận nhầm.

### Cách 1 chết ở đâu? — một vị trí bị thử đi thử lại

Xét từ "bẫy" `w = "aaaa...ab"` (k chữ `a` rồi một chữ `b`) với danh sách `["a", "aa", "aaa", w]`. Không có từ nào chứa `b` (ngoài chính `w`) → `w` **không** là từ ghép. Nhưng Cách 1 sẽ thử **mọi** cách cắt dãy chữ `a` thành `"a"/"aa"/"aaa"`, và lần nào cũng chết khi chạm `b`. Cùng một vị trí `start` bị ghé lại vô số lần, mỗi lần lại thử lại từ đầu. Ta đếm số lời gọi đệ quy:

In [12]:
def count_calls_brute(w: str, words: List[str]) -> int:
    word_set = set(words)
    L = len(w)
    calls = 0

    def can_split(start: int, count: int) -> bool:
        nonlocal calls          # nonlocal: sửa biến calls của hàm bên ngoài
        calls += 1
        if start == L:
            return count >= 2
        for end in range(start + 1, L + 1):
            if w[start:end] in word_set and can_split(end, count + 1):
                return True
        return False

    can_split(0, 0)
    return calls

print(" k | độ dài w | số lời gọi can_split (đáp án luôn là False)")
for k in [5, 10, 15, 20, 25]:
    w = "a" * k + "b"
    print(f"{k:>2} | {len(w):>8} | {count_calls_brute(w, ['a', 'aa', 'aaa', w]):>10}")

 k | độ dài w | số lời gọi can_split (đáp án luôn là False)
 5 |        6 |         29
10 |       11 |        601
15 |       16 |      12641
20 |       21 |     266080


25 |       26 |    5600911


Số lời gọi tăng **gấp khoảng 1.84 lần** mỗi khi thêm 1 chữ `a` (đây là dãy "Tribonacci": số cách cắt dãy `a` thành các mảnh dài 1, 2, 3). Với `k = 29` (từ dài 30 — vẫn hợp lệ theo đề) con số này lên tới hàng chục triệu cho **một** từ, và đề có tới `10^4` từ.

**Độ phức tạp Cách 1:** gọi `L` là độ dài từ dài nhất, `N` là số từ. Mỗi từ có thể tốn `O(2^L)` lời gọi, mỗi lời gọi lặp `L` giá trị `end` và cắt chuỗi `O(L)`:

- **Time:** `O(N · 2^L · L²)` trường hợp xấu nhất — quá chậm.
- **Space:** `O(S)` cho `set` (với `S` = tổng độ dài các từ) + `O(L)` độ sâu đệ quy.

Dù vậy Cách 1 **rất đáng giữ**: nó đơn giản, đúng theo định nghĩa → dùng làm **"trọng tài"** để kiểm tra các cách nhanh hơn trên input nhỏ (mục 16).

## 7. Cách 2 — Word Break DP cho từng từ (cách chính)

### Cách 1 vướng ở đâu?

Trong bẫy `"aaa...ab"`, câu hỏi "**từ vị trí `start` trở đi có cắt tiếp được không?**" được hỏi lại hàng nghìn lần cho **cùng** một `start`. Nhưng câu trả lời **không phụ thuộc** ta đã đi tới `start` bằng đường nào (cắt `"a"+"aa"` hay `"aa"+"a"` thì vẫn đứng ở cùng vị trí 3). Vậy hãy **tính mỗi vị trí một lần** và lưu vào bảng — đó chính là Word Break DP ở mục 3.

### Mấu chốt

Với mỗi từ `w` (độ dài `L`):

- `dp[i] = True` nếu tiền tố `w[:i]` cắt được thành các từ trong `words`, **không dùng mảnh là cả `w`**.
- `dp[0] = True`.
- `dp[i] = True` nếu có `j < i` với `dp[j] = True` và `w[j:i] ∈ word_set` — **trừ cặp** `j == 0 and i == L`.
- `w` là từ ghép ⟺ `dp[L] = True`.

Hai chi tiết "phòng thủ":

1. **Bỏ qua chuỗi rỗng**: nếu `L == 0` thì `dp[0] = True` sẽ khiến `""` bị nhận nhầm là từ ghép → `return False` ngay.
2. **Cấm cặp `(0, L)`** để không nhận "1 mảnh là chính nó" (mục 4).

Ta tách thành 2 hàm cho dễ đọc và dễ test: `is_concatenated_dp(w, word_set)` cho một từ, và `concat_dp(words)` chạy cho mọi từ.

Chạy `concat_dp` trên hai ví dụ của đề và vài trường hợp biên:

- `["a"]`: chỉ có 1 từ → không có gì để ghép → `[]`.
- `["a", "aa", "aaa"]`: `"aa" = "a"+"a"`, `"aaa" = "a"+"aa"` → `["aa", "aaa"]` (mảnh dùng lại được).
- `["", "a", "aa"]` (kiểu input của phiên bản đề cũ): chuỗi rỗng không được tính → `["aa"]`.

In [13]:
def is_concatenated_dp(w: str, word_set: set) -> bool:
    L = len(w)
    if L == 0:                          # chuỗi rỗng không bao giờ là từ ghép
        return False
    dp = [False] * (L + 1)
    dp[0] = True
    for i in range(1, L + 1):
        for j in range(i):
            if j == 0 and i == L:       # cấm mảnh duy nhất = cả từ w
                continue
            if dp[j] and w[j:i] in word_set:
                dp[i] = True
                break
    return dp[L]


def concat_dp(words: List[str]) -> List[str]:
    word_set = set(words)
    return [w for w in words if is_concatenated_dp(w, word_set)]


ws = set(examples[0][0])
for w in ["cat", "cats", "catsdogcats", "hippopotamuses", "ratcatdogcat"]:
    print(f"{w:<15} -> {is_concatenated_dp(w, ws)}")

for words, expected in examples:
    got = concat_dp(words)
    print(got)
    assert same_answer(got, expected)

assert concat_dp(["a"]) == []
assert same_answer(concat_dp(["a", "aa", "aaa"]), ["aa", "aaa"])
assert concat_dp(["", "a", "aa"]) == ["aa"]
assert same_answer(concat_dp(["cat", "catcat", "catcatcat", "dog"]), ["catcat", "catcatcat"])
print("Cách 2 PASS")

cat             -> False
cats            -> False
catsdogcats     -> True
hippopotamuses  -> False
ratcatdogcat    -> True
['catsdogcats', 'dogcatsdog', 'ratcatdogcat']
['catdog']
Cách 2 PASS


### Trace Cách 2: bảng `dp` của `"catsdogcats"`

Từ điển = mọi từ của Ví dụ 1. Chỉ số: `c0 a1 t2 s3 d4 o5 g6 c7 a8 t9 s10`, `L = 11`. Vòng `j` chạy **tăng dần** và dừng ở `j` đầu tiên thỏa:

| i | `w[:i]` | `j` đầu tiên thỏa | Mảnh cuối `w[j:i]` | `dp[i]` |
|---:|---|---:|---|:---:|
| 0 | `""` | — | điểm tựa | T |
| 3 | `"cat"` | 0 | `"cat"` | T |
| 4 | `"cats"` | 0 | `"cats"` | T |
| 7 | `"catsdog"` | 4 | `"dog"` (`j=3` thử `"sdog"` không được) | T |
| 10 | `"catsdogcat"` | 7 | `"cat"` | T |
| 11 | `"catsdogcats"` | 7 | `"cats"` (`j=0` bị cấm) | T |

Các ô còn lại `False`. Để **thấy được cách cắt**, hàm trace lưu thêm `parent[i] = j` (mảnh cuối của `w[:i]` bắt đầu ở `j`), rồi lần ngược từ `L` về `0`: `11 ← 7 ← 4 ← 0` → `"cats" + "dog" + "cats"`.

### Trace một từ **không** ghép được và một từ "suýt" ghép được

- `"hippopotamuses"`: không tiền tố nào là từ → không ô nào ngoài `dp[0]` bật lên. DP biết ngay mà không phải thử cây cắt nào.
- `"cats"`: `dp[3]` bật nhờ `"cat"`, nhưng `"s"` không phải từ; còn cặp `(0, 4)` — mảnh `"cats"` là cả từ — bị **cấm** → `dp[4] = False`. Không có dòng `continue` này, `"cats"` sẽ bị nhận nhầm.

In [14]:
def trace_concat_dp(w: str, word_set: set) -> bool:
    L = len(w)
    dp = [False] * (L + 1)
    parent = [-1] * (L + 1)
    dp[0] = True
    for i in range(1, L + 1):
        for j in range(i):
            if j == 0 and i == L:
                continue
            if dp[j] and w[j:i] in word_set:
                dp[i], parent[i] = True, j
                print(f"  dp[{i:>2}] = True  (j={j}, mảnh {w[j:i]!r})")
                break
    print("  chỉ số :", "".join(str(i % 10) for i in range(L + 1)))
    print("  bảng dp:", "".join("T" if x else "." for x in dp), "->", dp[L])
    if dp[L]:
        pieces, i = [], L
        while i > 0:                     # lần ngược parent để dựng lại cách cắt
            pieces.append(w[parent[i]:i])
            i = parent[i]
        print("  cách cắt:", " + ".join(reversed(pieces)))
    return dp[L]

ws = set(examples[0][0])
print("w = 'catsdogcats'")
trace_concat_dp("catsdogcats", ws)
print("w = 'ratcatdogcat'")
trace_concat_dp("ratcatdogcat", ws)

print("w = 'hippopotamuses'")
trace_concat_dp("hippopotamuses", ws)
print("w = 'cats'")
trace_concat_dp("cats", ws)

w = 'catsdogcats'
  dp[ 3] = True  (j=0, mảnh 'cat')
  dp[ 4] = True  (j=0, mảnh 'cats')
  dp[ 7] = True  (j=4, mảnh 'dog')
  dp[10] = True  (j=7, mảnh 'cat')
  dp[11] = True  (j=7, mảnh 'cats')
  chỉ số : 012345678901
  bảng dp: T..TT..T..TT -> True
  cách cắt: cats + dog + cats
w = 'ratcatdogcat'
  dp[ 3] = True  (j=0, mảnh 'rat')
  dp[ 6] = True  (j=3, mảnh 'cat')
  dp[ 9] = True  (j=6, mảnh 'dog')
  dp[12] = True  (j=9, mảnh 'cat')
  chỉ số : 0123456789012
  bảng dp: T..T..T..T..T -> True
  cách cắt: rat + cat + dog + cat
w = 'hippopotamuses'
  chỉ số : 012345678901234
  bảng dp: T.............. -> False
w = 'cats'
  dp[ 3] = True  (j=0, mảnh 'cat')
  chỉ số : 01234
  bảng dp: T..T. -> False


False

## 8. So sánh số thao tác: Cách 1 vs Cách 2 trên từ bẫy

Ta đếm số lần tra `set` (`piece in word_set`) — thao tác chiếm chủ yếu thời gian. Với DP, mỗi cặp `(j, i)` được xét **tối đa một lần**, nên số lần tra ≤ `L(L+1)/2`. Với từ dài 30: ≤ 465 lần, so với hàng triệu lời gọi của Cách 1.

In [15]:
def count_lookups_dp(w: str, word_set: set) -> int:
    L = len(w)
    lookups = 0
    dp = [False] * (L + 1)
    dp[0] = True
    for i in range(1, L + 1):
        for j in range(i):
            if j == 0 and i == L:
                continue
            if dp[j]:
                lookups += 1
                if w[j:i] in word_set:
                    dp[i] = True
                    break
    return lookups

print(" k | Cách 1: số lời gọi | Cách 2: số lần tra set")
for k in [5, 10, 15, 20, 25, 29]:
    w = "a" * k + "b"
    words_trap = ["a", "aa", "aaa", w]
    brute = count_calls_brute(w, words_trap) if k <= 20 else "(quá lâu, bỏ qua)"
    print(f"{k:>2} | {str(brute):>18} | {count_lookups_dp(w, set(words_trap)):>6}")

 k | Cách 1: số lời gọi | Cách 2: số lần tra set
 5 |                 29 |     13
10 |                601 |     48
15 |              12641 |    108
20 |             266080 |    193
25 |  (quá lâu, bỏ qua) |    303
29 |  (quá lâu, bỏ qua) |    409


## 9. Độ phức tạp Cách 2 — vì sao lúc thì nói `O(N·L²)`, lúc nói `O(N·L³)`?

Ký hiệu: `N` = số từ, `L` = độ dài từ dài nhất (≤ 30), `S` = tổng độ dài mọi từ (≤ `10^5`), `ℓ` = độ dài của từ đang xét.

Với một từ dài `ℓ`:

- Có khoảng `ℓ²/2` cặp `(j, i)`.
- Mỗi cặp làm `w[j:i]` (chép `i - j` ký tự, `O(ℓ)`) và băm chuỗi đó để tra `set` (cũng `O(ℓ)`).

→ **`O(ℓ³)` cho một từ**, tổng `O(Σ ℓ³) ≤ O(N · L³)`. Nếu coi việc cắt + băm chuỗi ngắn (≤ 30 ký tự) là `O(1)` thì người ta hay ghi gọn là `O(N · L²)`. Cả hai cách nói đều gặp; quan trọng là hiểu **thừa số `L` thứ ba đến từ đâu**: từ việc **tạo chuỗi con**. Cách 5 (Trie) sẽ bỏ hẳn thừa số này.

Ước lượng thực tế: vì `Σ ℓ ≤ 10^5` và `ℓ ≤ 30`, ta có `Σ ℓ³ ≤ 30² · Σ ℓ ≤ 9 × 10^7` phép chép ký tự trong trường hợp tệ nhất trên giấy — và phép cắt chuỗi của Python chạy bằng C nên rất nhanh. Cách 2 qua thoải mái giới hạn của LeetCode.

- **Space:** `O(S)` cho `set` + `O(L)` cho bảng `dp` (dùng lại cho từng từ).

### Tối ưu nhỏ nhưng hiệu quả: giới hạn độ dài mảnh

Một mảnh `w[j:i]` chỉ có thể là từ nếu độ dài `i - j` nằm trong `[min_len, max_len]` của các từ trong danh sách. Vậy vòng `j` chỉ cần chạy trong `[i - max_len, i - min_len]`. Ngoài ra ta dùng thủ thuật `discard`/`add` (mục 2.2) thay cho dòng `continue` — một cách khác để "loại chính nó".

In [16]:
def concat_dp_fast(words: List[str]) -> List[str]:
    word_set = set(w for w in words if w)          # bỏ chuỗi rỗng ngay từ đầu
    if not word_set:
        return []
    min_len = min(len(w) for w in word_set)
    max_len = max(len(w) for w in word_set)
    result = []
    for w in words:
        L = len(w)
        if L < 2 * min_len:                        # cần >= 2 mảnh, mỗi mảnh >= min_len
            continue
        word_set.discard(w)                        # tạm bỏ chính nó
        dp = [False] * (L + 1)
        dp[0] = True
        for i in range(min_len, L + 1):
            for j in range(max(0, i - max_len), i - min_len + 1):
                if dp[j] and w[j:i] in word_set:
                    dp[i] = True
                    break
        word_set.add(w)                            # thêm lại cho các từ sau
        if dp[L]:
            result.append(w)
    return result

for words, expected in examples:
    assert same_answer(concat_dp_fast(words), expected)
assert concat_dp_fast(["", "a", "aa"]) == ["aa"]
assert concat_dp_fast([""]) == []
assert same_answer(concat_dp_fast(["a", "aa", "aaa"]), ["aa", "aaa"])
print("concat_dp_fast PASS")

# Bao nhiêu cặp (j, i) được xét với từ dài 30, nếu min_len = 3, max_len = 5?
L, mn, mx = 30, 3, 5
full = L * (L + 1) // 2
bounded = sum(len(range(max(0, i - mx), i - mn + 1)) for i in range(mn, L + 1))
print(f"Không giới hạn: {full} cặp | Có giới hạn [{mn},{mx}]: {bounded} cặp")

concat_dp_fast PASS
Không giới hạn: 465 cặp | Có giới hạn [3,5]: 81 cặp


## 10. Cách 3 — Sắp theo độ dài, chỉ dùng các từ ngắn hơn

### Cách 2 còn gì chưa đẹp?

Cách 2 phải nhớ "loại chính nó" bằng `continue` hoặc `discard`/`add`. Ngoài ra `set` luôn chứa **mọi** từ, kể cả những từ **dài hơn** `w` — những từ này **không bao giờ** có thể là mảnh của `w`.

### Mấu chốt

Mọi mảnh ghép của `w` đều **ngắn hơn** `w` (đề nói rõ "shorter words"). Vậy:

1. Sắp `words` theo độ dài tăng dần.
2. Duyệt lần lượt; `known` (một `set`) chứa các từ **đã duyệt**, tức là các từ **không dài hơn** `w`.
3. Với `w`: chạy Word Break của `w` trên `known` (**chưa** thêm `w`). Nếu được → `w` là từ ghép.
4. **Sau đó** mới `known.add(w)` để các từ dài hơn dùng được.

Vì `w` chưa có trong `known` lúc kiểm tra, ta **không cần** cấm cặp `(0, L)` nữa. Còn những từ **cùng độ dài** với `w` đã nằm trong `known`? Không sao: một mảnh dài bằng cả `w` thì phải **bằng** `w`, mà các từ đều khác nhau → mảnh đó không có trong `known`.

Vẫn phải **bỏ qua chuỗi rỗng**: sau khi sắp xếp, `""` đứng đầu, và nếu kiểm tra nó thì `dp[0] = True` sẽ trả lời "có".

In [17]:
def word_break_in(w: str, known: set) -> bool:
    # Word Break thuần (bài 139) cho w với từ điển known
    L = len(w)
    dp = [False] * (L + 1)
    dp[0] = True
    for i in range(1, L + 1):
        for j in range(i):
            if dp[j] and w[j:i] in known:
                dp[i] = True
                break
    return dp[L]


def concat_sorted(words: List[str]) -> List[str]:
    known = set()
    result = []
    for w in sorted(words, key=len):
        if not w:                        # bỏ qua chuỗi rỗng
            continue
        if known and word_break_in(w, known):   # known rỗng thì chắc chắn không ghép được
            result.append(w)
        known.add(w)                     # thêm SAU khi kiểm tra
    return result

for words, expected in examples:
    got = concat_sorted(words)
    print(got)
    assert same_answer(got, expected)
assert concat_sorted(["", "a", "aa"]) == ["aa"]
assert concat_sorted(["a"]) == []
print("Cách 3 PASS")

['dogcatsdog', 'catsdogcats', 'ratcatdogcat']
['catdog']
Cách 3 PASS


### Trace Cách 3: `known` lớn dần theo thứ tự độ dài

Hàm `trace_sorted` in, cho mỗi từ theo thứ tự đã sắp: kích thước `known` **trước khi** kiểm tra, kết quả, rồi thêm từ vào `known`. Để ý: khi tới `"dogcatsdog"` (dài 10), `known` chỉ có 4 từ ngắn `cat, dog, rat, cats` — đủ để ghép. Và `"catsdogcats"` được kiểm tra khi `known` đã có cả `"dogcatsdog"` — từ dài hơn đó vô hại vì không thể là mảnh.

In [18]:
def trace_sorted(words: List[str]) -> List[str]:
    known = set()
    result = []
    print(f"{'từ':<15} {'len':>3} | {'|known| trước':>13} | ghép?")
    for w in sorted(words, key=len):
        if not w:
            print(f"{repr(w):<15} {0:>3} | {'':>13} | bỏ qua chuỗi rỗng")
            continue
        ok = bool(known) and word_break_in(w, known)
        print(f"{w:<15} {len(w):>3} | {len(known):>13} | {'✔' if ok else '✘'}")
        if ok:
            result.append(w)
        known.add(w)
    print("Kết quả:", result)
    return result

trace_sorted(examples[0][0])

từ              len | |known| trước | ghép?
cat               3 |             0 | ✘
dog               3 |             1 | ✘
rat               3 |             2 | ✘
cats              4 |             3 | ✘
dogcatsdog       10 |             4 | ✔
catsdogcats      11 |             5 | ✔
ratcatdogcat     12 |             6 | ✔
hippopotamuses   14 |             7 | ✘
Kết quả: ['dogcatsdog', 'catsdogcats', 'ratcatdogcat']


['dogcatsdog', 'catsdogcats', 'ratcatdogcat']

**Độ phức tạp Cách 3:** sắp xếp `O(N log N)` (so sánh theo độ dài là `O(1)`), cộng phần Word Break cho từng từ giống Cách 2: `O(Σ ℓ³) ≤ O(N · L³)`. **Space:** `O(S)`.

Trên thực tế Cách 3 thường **nhanh hơn một chút** vì `known` nhỏ hơn ở các từ đầu, và ý tưởng "chỉ dùng nguyên liệu ngắn hơn đã có" rất dễ giải thích trong phỏng vấn. Nhược điểm: kết quả ra theo **thứ tự độ dài**, không theo thứ tự input — vẫn hợp lệ vì đề cho phép thứ tự bất kỳ.

Cũng có thể kết hợp Cách 3 với tối ưu giới hạn độ dài mảnh của mục 9 (chính là Bài 4 ở cuối notebook).

## 11. Kiến thức nền: memoization (ghi nhớ kết quả)

**Memoization** = trước khi tính `f(x)`, xem đã từng tính chưa; nếu rồi thì trả kết quả đã lưu; nếu chưa thì tính, **lưu vào memo**, rồi trả về. Đây là "DP từ trên xuống" (top-down): viết như đệ quy, nhưng mỗi bài con chỉ giải một lần.

### Ví dụ mini: Fibonacci

`fib(n) = fib(n-1) + fib(n-2)`, `fib(0) = 0`, `fib(1) = 1`. Đệ quy thuần gọi `fib(n-2)` hai lần (một lần trực tiếp, một lần bên trong `fib(n-1)`), cứ thế nhân lên → số lời gọi tăng theo hàm mũ.

### `functools.lru_cache`: memo tự động

`@lru_cache(maxsize=None)` đặt ngay trên `def` là một **decorator** (lớp bọc): Python tự lưu kết quả theo **tham số**. Tham số phải **hashable** (băm được) như `int`, `str`, `tuple` — ở Cách 4 tham số là một **chuỗi**, rất hợp. Hàm được bọc còn có `.cache_info()` để xem số lần "trúng" bộ nhớ (`hits`) và số lần phải tính (`misses`).

In [19]:
from functools import lru_cache

calls_plain = 0
def fib_plain(n: int) -> int:
    global calls_plain
    calls_plain += 1
    return n if n < 2 else fib_plain(n - 1) + fib_plain(n - 2)

@lru_cache(maxsize=None)
def fib_memo(n: int) -> int:
    return n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)

print("fib_plain(25) =", fib_plain(25), "| số lời gọi:", calls_plain)
print("fib_memo(25)  =", fib_memo(25), "|", fib_memo.cache_info())

fib_plain(25) = 75025 | số lời gọi: 242785
fib_memo(25)  = 75025 | CacheInfo(hits=23, misses=26, maxsize=None, currsize=26)


## 12. Cách 4 — DFS + memo theo chuỗi (lời giải "5 dòng")

### Đổi góc nhìn: hỏi trực tiếp về **chuỗi**, không về vị trí

Định nghĩa `dfs(s)` = "`s` có cắt được thành **≥ 2** từ trong `word_set` không?". Ta thử mọi điểm cắt `i` từ `1` tới `len(s) - 1`, chia `s` thành `prefix = s[:i]` và `suffix = s[i:]` (**cả hai khác rỗng, cả hai ngắn hơn `s`**):

- `prefix` phải là một từ, **và**
- `suffix` hoặc **là một từ** (tổng cộng 2 mảnh), hoặc **tự nó** cắt được thành ≥ 2 từ (`dfs(suffix)`, tổng cộng ≥ 3 mảnh).

```python
@lru_cache(maxsize=None)
def dfs(s):
    for i in range(1, len(s)):
        prefix, suffix = s[:i], s[i:]
        if prefix in word_set and (suffix in word_set or dfs(suffix)):
            return True
    return False
```

### Vì sao cách này **tự động** đúng các chi tiết khó?

- **≥ 2 mảnh:** `i` chạy từ 1 tới `len(s) - 1` nên luôn có ít nhất `prefix` và `suffix` → không bao giờ "1 mảnh là chính nó". Không cần `discard` hay `continue`.
- **Chuỗi rỗng:** `dfs("")` → `range(1, 0)` rỗng → `False`. Và `prefix`, `suffix` không bao giờ rỗng nên `""` (nếu có trong `set`) không bao giờ được dùng làm mảnh.
- **Memo dùng chung giữa các từ:** `word_set` cố định, nên `dfs("cats")` có cùng đáp án dù được hỏi từ `"catsdogcats"` hay từ `"dogcats"`. Ta tạo `dfs` **một lần** cho cả danh sách.

In [20]:
def concat_dfs_memo(words: List[str]) -> List[str]:
    word_set = set(words)

    @lru_cache(maxsize=None)
    def dfs(s: str) -> bool:
        for i in range(1, len(s)):
            prefix, suffix = s[:i], s[i:]
            if prefix in word_set and (suffix in word_set or dfs(suffix)):
                return True
        return False

    return [w for w in words if dfs(w)]

for words, expected in examples:
    got = concat_dfs_memo(words)
    print(got)
    assert same_answer(got, expected)
assert concat_dfs_memo(["", "a", "aa"]) == ["aa"]
assert concat_dfs_memo(["a"]) == []
print("Cách 4 PASS")

['catsdogcats', 'dogcatsdog', 'ratcatdogcat']
['catdog']
Cách 4 PASS


### Trace Cách 4: thấy memo được dùng lại (♻)

Để thấy memo phát huy, ta dùng danh sách `["a", "aa", "aaab"]`. `"aaab"` không ghép được (không từ nào chứa `b`), nhưng trong lúc tìm, các hậu tố `"aab"`, `"ab"`, `"b"` được hỏi tới từ nhiều đường khác nhau. Hàm `trace_dfs` dùng `dict` memo tự viết để in rõ: **tính mới** (thụt lề theo độ sâu) hay **dùng lại** (♻).

Để ý `dfs` chỉ được gọi khi `prefix` là từ **và** `suffix` **không** phải từ — đúng thứ tự của biểu thức `prefix in word_set and (suffix in word_set or dfs(suffix))` (Python **đánh giá ngắn mạch — short-circuit**: `and` dừng khi vế trái `False`, `or` dừng khi vế trái `True`).

In [21]:
def trace_dfs(words: List[str]) -> List[str]:
    word_set = set(words)
    memo = {}

    def dfs(s: str, depth: int) -> bool:
        pad = "    " * depth
        if s in memo:
            print(f"{pad}♻ dfs({s!r}) = {memo[s]} (dùng lại memo)")
            return memo[s]
        print(f"{pad}dfs({s!r}) tính mới")
        ans = False
        for i in range(1, len(s)):
            prefix, suffix = s[:i], s[i:]
            if prefix in word_set:
                if suffix in word_set:
                    print(f"{pad}  {prefix!r} + {suffix!r}: cả hai là từ ✔")
                    ans = True
                    break
                print(f"{pad}  {prefix!r} là từ, {suffix!r} không phải từ -> hỏi dfs({suffix!r})")
                if dfs(suffix, depth + 1):
                    ans = True
                    break
        memo[s] = ans
        print(f"{pad}=> dfs({s!r}) = {ans}")
        return ans

    result = [w for w in words if dfs(w, 0)]
    print("Kết quả:", result)
    return result

trace_dfs(["a", "aa", "aaab"])

dfs('a') tính mới
=> dfs('a') = False
dfs('aa') tính mới
  'a' + 'a': cả hai là từ ✔
=> dfs('aa') = True
dfs('aaab') tính mới
  'a' là từ, 'aab' không phải từ -> hỏi dfs('aab')
    dfs('aab') tính mới
      'a' là từ, 'ab' không phải từ -> hỏi dfs('ab')
        dfs('ab') tính mới
          'a' là từ, 'b' không phải từ -> hỏi dfs('b')
            dfs('b') tính mới
            => dfs('b') = False
        => dfs('ab') = False
      'aa' là từ, 'b' không phải từ -> hỏi dfs('b')
        ♻ dfs('b') = False (dùng lại memo)
    => dfs('aab') = False
  'aa' là từ, 'ab' không phải từ -> hỏi dfs('ab')
    ♻ dfs('ab') = False (dùng lại memo)
=> dfs('aaab') = False
Kết quả: ['aa']


['aa']

### Đọc trace bằng bảng (theo thứ tự memo được **lưu**)

| Chuỗi `s` | Đáp án `dfs(s)` | Lý do |
|---|:---:|---|
| `"a"` | F | Không có điểm cắt nào (`len = 1`) |
| `"aa"` | T | `"a" + "a"` — cả hai là từ |
| `"b"` | F | Không có điểm cắt |
| `"ab"` | F | `"a"` là từ nhưng `"b"` không ghép được |
| `"aab"` | F | `"a"`+`"ab"` ✘; `"aa"`+`"b"` → `dfs("b")` ♻ đã biết F |
| `"aaab"` | F | `"a"`+`"aab"` ✘; `"aa"`+`"ab"` → `dfs("ab")` ♻ đã biết F |

Hai lần ♻ chính là chỗ Cách 1 phải đi lại cả cây con. Cell dưới đếm số lần **thực sự tính** `dfs` trên từ bẫy `"a"*k + "b"` để so với Cách 1: memo khiến mỗi chuỗi chỉ tính một lần → đúng `k + 4` lần (4 từ trong danh sách + `k` hậu tố `"b"`, `"ab"`, ..., `"a"*(k-1) + "b"`), tăng **tuyến tính** thay vì hàm mũ.

In [22]:
def count_dfs_misses(words: List[str]) -> int:
    word_set = set(words)

    @lru_cache(maxsize=None)
    def dfs(s: str) -> bool:
        for i in range(1, len(s)):
            prefix, suffix = s[:i], s[i:]
            if prefix in word_set and (suffix in word_set or dfs(suffix)):
                return True
        return False

    for w in words:
        dfs(w)
    return dfs.cache_info().misses       # số lần phải tính mới

print(" k | Cách 1: lời gọi | Cách 4: số lần tính dfs")
for k in [5, 10, 15, 20]:
    w = "a" * k + "b"
    trap = ["a", "aa", "aaa", w]
    print(f"{k:>2} | {count_calls_brute(w, trap):>15} | {count_dfs_misses(trap):>6}")

 k | Cách 1: lời gọi | Cách 4: số lần tính dfs
 5 |              29 |      9
10 |             601 |     14
15 |           12641 |     19


20 |          266080 |     24


**Độ phức tạp Cách 4:** memo chỉ chứa **hậu tố** của các từ (vì `dfs` chỉ được gọi trên `suffix`), nên có tối đa `S` khóa. Mỗi lần tính `dfs(s)` lặp `len(s)` điểm cắt, mỗi điểm cắt tạo 2 chuỗi con `O(len(s))`:

- **Time:** `O(Σ ℓ³) ≤ O(N · L³)` — cùng bậc với Cách 2.
- **Space:** memo lưu các chuỗi hậu tố làm khóa: `O(Σ ℓ²) ≤ O(S · L)` ký tự, cộng `O(L)` độ sâu đệ quy.

Ưu điểm: code **ngắn nhất**, tự xử lý "≥ 2 mảnh" và chuỗi rỗng. Nhược điểm: tốn bộ nhớ hơn Cách 2 (lưu chuỗi làm khóa), và đệ quy (an toàn ở đây vì độ sâu ≤ 30).

## 13. Kiến thức nền: Trie (cây tiền tố)

### Mọi cách trên vẫn tốn công ở đâu?

Ở Cách 2, với một vị trí bắt đầu `j`, ta tạo lần lượt `w[j:j+1]`, `w[j:j+2]`, `w[j:j+3]`, ... và **mỗi** chuỗi con đều phải chép ký tự rồi băm lại **từ đầu** — dù chúng chỉ hơn nhau đúng 1 ký tự. Đó là thừa số `L` thứ ba.

### Trie là gì?

**Trie** (đọc là "trai", còn gọi **prefix tree — cây tiền tố**) lưu một tập từ dưới dạng cây:

- Mỗi **nút** có các **cạnh con** gắn nhãn một ký tự (ta dùng `dict`: ký tự → nút con).
- Đi từ gốc theo các ký tự `c`, `a`, `t` là đi theo tiền tố `"cat"`.
- Nút nào là **điểm kết thúc của một từ** thì đánh dấu `is_end = True`.

Lợi ích: đứng ở vị trí `j` của `w`, ta **đi dần** xuống trie theo `w[j]`, `w[j+1]`, ... mỗi bước `O(1)`. Mỗi khi gặp nút `is_end`, ta biết ngay `w[j:k+1]` là một từ — **không cần tạo chuỗi con**. Gặp ký tự không có cạnh con → **dừng sớm** vì không từ nào có tiền tố đó.

### Ví dụ mini: trie của `["cat", "cats", "dog"]`

```text
(gốc)
 ├─ c ─ a ─ t* ─ s*
 └─ d ─ o ─ g*
```

(`*` = `is_end`). Đi trong chuỗi `"catsdog"` từ vị trí 0: `c → a → t*` (gặp từ `"cat"`, kết thúc ở chỉ số 2) → `s*` (gặp `"cats"`, chỉ số 3) → ký tự `d` không có con → dừng.

In [23]:
class TrieNode:
    __slots__ = ("children", "is_end")      # __slots__: tiết kiệm bộ nhớ cho nhiều nút

    def __init__(self):
        self.children = {}                  # ký tự -> TrieNode
        self.is_end = False


def build_trie(words: List[str]) -> TrieNode:
    root = TrieNode()
    for w in words:
        node = root
        for ch in w:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True
    return root


def print_trie(node: TrieNode, prefix: str = "", depth: int = 0) -> None:
    for ch, child in sorted(node.children.items()):
        print("  " * depth + ch + ("*" if child.is_end else ""), f"  (tiền tố {prefix + ch!r})")
        print_trie(child, prefix + ch, depth + 1)


root = build_trie(["cat", "cats", "dog"])
print_trie(root)

# Đi trong "catsdog" từ vị trí j = 0: liệt kê mọi từ bắt đầu ở j
s, j = "catsdog", 0
node = root
for k in range(j, len(s)):
    node = node.children.get(s[k])          # .get trả None nếu không có cạnh con
    if node is None:
        print(f"k={k}: ký tự {s[k]!r} không có cạnh con -> dừng sớm")
        break
    if node.is_end:
        print(f"k={k}: gặp từ {s[j:k + 1]!r} (chỉ in ra để xem, thuật toán không cần cắt chuỗi)")

c   (tiền tố 'c')
  a   (tiền tố 'ca')
    t*   (tiền tố 'cat')
      s*   (tiền tố 'cats')
d   (tiền tố 'd')
  o   (tiền tố 'do')
    g*   (tiền tố 'dog')
k=2: gặp từ 'cat' (chỉ in ra để xem, thuật toán không cần cắt chuỗi)
k=3: gặp từ 'cats' (chỉ in ra để xem, thuật toán không cần cắt chuỗi)
k=4: ký tự 'd' không có cạnh con -> dừng sớm


## 14. Cách 5 — Word Break DP + Trie (tùy chọn)

Giữ nguyên ý tưởng Cách 2 (bảng `dp` theo vị trí), nhưng đổi **chiều** cập nhật: thay vì "với mỗi `i`, tìm `j`", ta làm "**với mỗi `j` đã đến được** (`dp[j] = True`), đi xuống trie từ `w[j]` để **bật** mọi `dp[k+1]` mà `w[j:k+1]` là từ".

```python
for j in range(L):
    if not dp[j]:
        continue
    node = root
    for k in range(j, L):
        node = node.children.get(w[k])
        if node is None:
            break                                # không từ nào có tiền tố này
        if node.is_end and not (j == 0 and k == L - 1):   # cấm mảnh = cả w
            dp[k + 1] = True
```

Trie chứa **mọi** từ, nên ta vẫn cấm mảnh phủ cả từ (cặp `j == 0`, `k == L - 1`), giống Cách 2. Chuỗi rỗng: chèn `""` vào trie chỉ đánh dấu `root.is_end` — mà ta không bao giờ kiểm tra `root.is_end` (luôn đi xuống ít nhất 1 ký tự) → `""` không bao giờ được dùng làm mảnh; còn bản thân `""` thì ta bỏ qua bằng `if L == 0`.

**Độ phức tạp:** mỗi `j` đi xuống trie tối đa `ℓ` bước `O(1)` → `O(ℓ²)` cho một từ, tổng **`O(Σ ℓ²) ≤ O(N · L²)`** — đã bỏ được thừa số cắt chuỗi. Xây trie `O(S)`. **Space:** `O(S)` nút trie (mỗi nút một `dict`, nên hằng số bộ nhớ lớn hơn `set`).

In [24]:
def concat_trie(words: List[str]) -> List[str]:
    root = build_trie(words)
    result = []
    for w in words:
        L = len(w)
        if L == 0:
            continue
        dp = [False] * (L + 1)
        dp[0] = True
        for j in range(L):
            if not dp[j]:
                continue
            node = root
            for k in range(j, L):
                node = node.children.get(w[k])
                if node is None:
                    break
                if node.is_end and not (j == 0 and k == L - 1):
                    dp[k + 1] = True
            if dp[L]:                       # đã phủ kín -> dừng sớm
                break
        if dp[L]:
            result.append(w)
    return result

for words, expected in examples:
    got = concat_trie(words)
    print(got)
    assert same_answer(got, expected)
assert concat_trie(["", "a", "aa"]) == ["aa"]
assert concat_trie(["a"]) == []
print("Cách 5 PASS")

['catsdogcats', 'dogcatsdog', 'ratcatdogcat']
['catdog']
Cách 5 PASS


## 15. Chuyển sang format LeetCode: `class Solution`

Ta nộp **Cách 2** (Word Break DP cho từng từ) kèm tối ưu giới hạn độ dài mảnh — vì nó dễ giải thích nhất, không đệ quy, bộ nhớ nhỏ, và nhanh. Những điểm cần giữ khi chuyển sang class:

- `from typing import List` (LeetCode đã import sẵn, nhưng khi tự chạy ta cần).
- Tên method đúng: `findAllConcatenatedWordsInADict(self, words)`.
- `set` được tạo **bên trong** method (mỗi test case một `set` mới).
- Bỏ chuỗi rỗng khỏi `set` và không kiểm tra chuỗi rỗng.

Test bao gồm: 2 ví dụ đề, danh sách 1 từ, mảnh dùng lại (`"catcatcat"`), chuỗi rỗng (phiên bản cũ), từ bẫy dài 30 (`"a"*29 + "b"` — không ghép được, nhưng DP vẫn trả lời ngay), và một từ chỉ ghép được nếu **chịu quay lại thử mảnh khác** (`"catsdogcats"`: lấy mảnh ngắn nhất `"cat"` trước sẽ vào ngõ cụt).

In [25]:
class Solution:
    def findAllConcatenatedWordsInADict(self, words: List[str]) -> List[str]:
        word_set = set(w for w in words if w)
        if not word_set:
            return []
        min_len = min(len(w) for w in word_set)
        max_len = max(len(w) for w in word_set)
        result = []
        for w in words:
            L = len(w)
            if L < 2 * min_len:              # cũng loại luôn chuỗi rỗng
                continue
            dp = [False] * (L + 1)
            dp[0] = True
            for i in range(min_len, L + 1):
                for j in range(max(0, i - max_len), i - min_len + 1):
                    if j == 0 and i == L:    # cấm mảnh duy nhất = cả w
                        continue
                    if dp[j] and w[j:i] in word_set:
                        dp[i] = True
                        break
            if dp[L]:
                result.append(w)
        return result


sol = Solution()
for words, expected in examples:
    assert same_answer(sol.findAllConcatenatedWordsInADict(words), expected)

assert sol.findAllConcatenatedWordsInADict(["a"]) == []
assert sol.findAllConcatenatedWordsInADict(["cat", "dog"]) == []
assert same_answer(sol.findAllConcatenatedWordsInADict(["cat", "catcat", "catcatcat"]), ["catcat", "catcatcat"])
assert sol.findAllConcatenatedWordsInADict(["", "a", "aa"]) == ["aa"]
assert sol.findAllConcatenatedWordsInADict([""]) == []
trap = "a" * 29 + "b"
assert sol.findAllConcatenatedWordsInADict(["a", "aa", "aaa", trap]) == ["aa", "aaa"]
# "catsdogcats": chọn "cat" (mảnh ngắn nhất) trước sẽ ngõ cụt -> phải thử "cats"
assert sol.findAllConcatenatedWordsInADict(["cat", "cats", "dog", "catsdogcats"]) == ["catsdogcats"]
print("class Solution: mọi test PASS")

class Solution: mọi test PASS


### Bản nộp thay thế: DFS + memo (Cách 4)

Nếu thích ngắn gọn, bản Cách 4 cũng qua LeetCode. Lưu ý đặt `lru_cache` lên **hàm con** định nghĩa **bên trong** method (không đặt lên chính method có `self`), để memo được tạo mới cho mỗi lần gọi và đi kèm đúng `word_set` của lần gọi đó.

In [26]:
class SolutionDFS:
    def findAllConcatenatedWordsInADict(self, words: List[str]) -> List[str]:
        word_set = set(words)

        @lru_cache(maxsize=None)
        def dfs(s: str) -> bool:
            for i in range(1, len(s)):
                prefix, suffix = s[:i], s[i:]
                if prefix in word_set and (suffix in word_set or dfs(suffix)):
                    return True
            return False

        return [w for w in words if dfs(w)]


sol2 = SolutionDFS()
for words, expected in examples:
    assert same_answer(sol2.findAllConcatenatedWordsInADict(words), expected)
assert sol2.findAllConcatenatedWordsInADict(["", "a", "aa"]) == ["aa"]
assert sol2.findAllConcatenatedWordsInADict(["a", "aa", "aaa", "a" * 29 + "b"]) == ["aa", "aaa"]
print("class SolutionDFS: mọi test PASS")

class SolutionDFS: mọi test PASS


## 16. Test đối chiếu chéo trên input ngẫu nhiên

Ví dụ đề chỉ có 2 ca — quá ít để tin tưởng. Ta sinh **500 danh sách ngẫu nhiên** nhỏ (bảng chữ cái chỉ `"ab"` để các từ dễ "đụng" nhau, tạo nhiều từ ghép), độ dài từ 1–7, **có lúc chèn thêm chuỗi rỗng** để kiểm tra phòng thủ. Mọi cách phải cho **cùng tập kết quả** với **trọng tài** Cách 1 (brute force). `random.seed(472)` cố định để lần chạy nào cũng ra cùng dữ liệu.

In [27]:
import random

random.seed(472)
methods = {
    "Cách 2 DP": concat_dp,
    "Cách 2 DP fast": concat_dp_fast,
    "Cách 3 sorted": concat_sorted,
    "Cách 4 DFS memo": concat_dfs_memo,
    "Cách 5 Trie": concat_trie,
    "Solution": Solution().findAllConcatenatedWordsInADict,
    "SolutionDFS": SolutionDFS().findAllConcatenatedWordsInADict,
}

total_concat = 0
for t in range(500):
    pool = set()
    for _ in range(random.randint(1, 12)):
        pool.add("".join(random.choice("ab") for _ in range(random.randint(1, 7))))
    words = list(pool)
    if random.random() < 0.2:
        words.append("")                     # thỉnh thoảng có chuỗi rỗng (phiên bản đề cũ)
    random.shuffle(words)
    expected = sorted(concat_brute(words))
    total_concat += len(expected)
    for name, f in methods.items():
        got = sorted(f(list(words)))
        assert got == expected, (name, words, got, expected)

print("500 test ngẫu nhiên: mọi cách khớp với brute force")
print("Tổng số từ ghép gặp trong các test:", total_concat)

500 test ngẫu nhiên: mọi cách khớp với brute force
Tổng số từ ghép gặp trong các test: 910


## 17. Thử với input cỡ lớn (gần giới hạn đề)

Ta tạo một input "nặng": 30 từ `"a"`, `"aa"`, ..., `"a"*30` (mọi từ từ độ dài 2 trở lên đều là từ ghép), cộng 3000 từ ngẫu nhiên dài 10–30 trên bảng chữ cái `"abc"`, và thêm 1000 từ "bẫy" dạng `"a"*k + "b" + ...`. Tổng độ dài vẫn dưới `10^5`.

Cell đo thời gian chạy của các cách nhanh (bỏ brute force vì có thể rất lâu với từ bẫy). **Con số thời gian phụ thuộc máy**, nên ta không `assert` thời gian, chỉ kiểm tra các cách ra cùng kết quả và quan sát bậc độ lớn.

In [28]:
import time

random.seed(2024)
big = {"a" * k for k in range(1, 31)}
while len(big) < 30 + 3000:
    big.add("".join(random.choice("abc") for _ in range(random.randint(10, 30))))
while len(big) < 30 + 3000 + 1000:
    k = random.randint(10, 25)
    big.add("a" * k + "b" + "".join(random.choice("ab") for _ in range(random.randint(0, 29 - k))))
big = list(big)
print("Số từ:", len(big), "| tổng độ dài:", sum(map(len, big)))

reference = None
for name in ["Cách 2 DP", "Cách 2 DP fast", "Cách 3 sorted", "Cách 4 DFS memo", "Cách 5 Trie", "Solution"]:
    t0 = time.perf_counter()
    res = sorted(methods[name](big))
    dt = time.perf_counter() - t0
    if reference is None:
        reference = res
    assert res == reference
    print(f"{name:<16}: {len(res):>4} từ ghép | {dt * 1000:8.1f} ms")

Số từ: 4030 | tổng độ dài: 85515


Cách 2 DP       :  533 từ ghép |     42.9 ms
Cách 2 DP fast  :  533 từ ghép |     33.7 ms


Cách 3 sorted   :  533 từ ghép |     26.3 ms


Cách 4 DFS memo :  533 từ ghép |     33.8 ms
Cách 5 Trie     :  533 từ ghép |     38.0 ms


Solution        :  533 từ ghép |     39.5 ms


## 18. Vì sao các cách đúng? (invariant bằng lời)

### Cách 2 (DP theo vị trí)

**Invariant:** sau khi xử lý xong `i`, `dp[i] = True` **khi và chỉ khi** tiền tố `w[:i]` cắt được thành các từ trong `words`, với điều kiện không dùng mảnh là cả `w`.

- **Không bỏ sót:** nếu `w[:i]` cắt được, gọi mảnh cuối là `w[j:i]`. Khi đó `w[:j]` cũng cắt được (bỏ mảnh cuối đi) và `j < i`, nên theo invariant `dp[j] = True` đã có từ trước. Vòng `j` sẽ gặp cặp `(j, i)` này (hoặc một `j` khác cũng thỏa) → `dp[i] = True`.
- **Không nhận nhầm:** `dp[i]` chỉ bật khi có `dp[j] = True` **và** `w[j:i]` là từ — tức là ghép được một cách cắt thật của `w[:j]` với một từ thật.
- **≥ 2 mảnh:** mảnh nào cũng khác rỗng; cặp `(0, L)` bị cấm nên không mảnh nào dài bằng `w` → cần ít nhất 2 mảnh để phủ `L` ký tự.

`break` không làm sai vì `dp[i]` chỉ cần **một** cách — các `j` khác không đổi được giá trị `True`.

### Cách 3 (sắp theo độ dài)

**Invariant:** lúc kiểm tra `w`, `known` = tập các từ (khác rỗng) **đã duyệt**, gồm **mọi** từ ngắn hơn `w` và **không** chứa `w`. Mọi mảnh hợp lệ của `w` đều ngắn hơn `w` nên đều có trong `known`; còn mảnh dài bằng `w` thì không thể có trong `known`. Vậy "Word Break của `w` trên `known`" đúng bằng câu hỏi của đề.

### Cách 4 (DFS + memo)

**Invariant:** `dfs(s) = True` ⟺ `s` cắt được thành **≥ 2** từ. Chứng minh theo độ dài `s`: một cách cắt ≥ 2 mảnh có mảnh đầu `prefix` (là từ) và phần sau `suffix` khác rỗng; `suffix` hoặc là 1 từ, hoặc cắt được thành ≥ 2 từ (= `dfs(suffix)` với chuỗi ngắn hơn, đúng theo giả thiết quy nạp). Memo đúng vì `word_set` không đổi trong suốt quá trình.

### Cách 5 (Trie)

Cùng invariant với Cách 2, chỉ khác chiều cập nhật: mọi `dp[k+1]` được bật từ một `j` đã `True` qua một từ `w[j:k+1]` mà trie xác nhận. Khi vòng ngoài tới `j`, mọi ô `dp[j']` với `j' < j` đã được dùng để bật các ô phía sau, nên `dp[j]` đã có giá trị cuối cùng.

## 19. Bảng so sánh các cách

Ký hiệu: `N` = số từ, `L` = độ dài từ dài nhất (≤ 30), `S` = tổng độ dài (≤ `10^5`), `ℓ` = độ dài một từ.

| Cách | Ý tưởng | Time | Space | "≥ 2 mảnh" xử lý bằng | Nhận xét |
|---|---|---|---|---|---|
| 1. Brute force | Đệ quy thử mọi cách cắt, đếm mảnh | `O(N · 2^L · L²)` xấu nhất | `O(S + L)` | biến `count >= 2` | Đúng theo định nghĩa; chỉ dùng làm trọng tài |
| **2. Word Break DP** | `dp[i]` cho từng từ | `O(Σ ℓ³) ≤ O(N · L³)` | `O(S + L)` | cấm cặp `(0, L)` / `discard` | **Cách chính**, dễ giải thích, không đệ quy |
| 2'. DP + giới hạn độ dài | `j ∈ [i - max_len, i - min_len]` | như trên, thường nhanh hơn nhiều | `O(S + L)` | như trên | Bản nộp trong `class Solution` |
| 3. Sắp theo độ dài | Chỉ dùng từ ngắn hơn đã thêm | `O(N log N + Σ ℓ³)` | `O(S + L)` | `w` chưa có trong `known` | Ý tưởng "nguyên liệu" rất trực quan |
| 4. DFS + memo | `dfs(s)`: prefix là từ và suffix là từ/ghép được | `O(Σ ℓ³)` | `O(S · L)` (khóa memo là chuỗi) | `1 <= i < len(s)` | Code ngắn nhất; tự lo chuỗi rỗng |
| 5. DP + Trie | Đi xuống trie thay vì cắt chuỗi | `O(S + Σ ℓ²) ≤ O(N · L²)` | `O(S)` nút trie | cấm `j == 0, k == L-1` | Bỏ thừa số cắt chuỗi; code dài hơn |

Với giới hạn của đề (`L ≤ 30`), các Cách 2–5 đều chạy tốt; khác biệt `L²` vs `L³` chỉ là thừa số ≤ 30 và trong Python, phép cắt chuỗi viết bằng C thường "bù" lại chi phí của các bước Python thuần khi đi trie.

## 20. Các lỗi người mới hay gặp

### Lỗi 1: Quên loại chính từ đó → mọi từ đều "là từ ghép"

Đã thấy ở mục 4: chạy Word Break với từ điển chứa cả `w` thì `dp[L]` luôn bật nhờ cặp `(0, L)`.

### Lỗi 2: Quên chuỗi rỗng

Với input có `""` (phiên bản đề cũ), `is_concatenated` không có dòng `if L == 0` sẽ trả `True` cho `""` vì `dp[0] = True` ngay từ đầu. Ở Cách 3, `""` đứng đầu sau khi sắp xếp, nên lỗi xuất hiện ngay từ từ đầu tiên.

### Lỗi 3: Tham lam (greedy) — luôn lấy mảnh khớp đầu tiên

Người mới hay viết: "đi từ trái sang, gặp tiền tố nào là từ thì **cắt luôn** rồi đi tiếp, không quay lại". Với `"catsdogcats"`, tham lam lấy `"cat"` (ngắn nhất) → còn `"sdogcats"` → bế tắc → kết luận sai `False`. Đổi sang lấy mảnh **dài nhất** cũng sai với ví dụ khác (vd `"aab"` với `["a", "aa", "ab"]`: dài nhất lấy `"aa"`, còn `"b"` → bế tắc, trong khi `"a" + "ab"` đúng). Phải **thử mọi** mảnh — đó là việc của DP.

Cell dưới chạy cả ba hàm sai cạnh đáp án đúng.

In [29]:
def wrong_no_exclude(words: List[str]) -> List[str]:
    word_set = set(words)
    return [w for w in words if word_break_in(w, word_set)]          # LỖI 1: không loại chính w


def wrong_empty(words: List[str]) -> List[str]:
    known, result = set(), []
    for w in sorted(words, key=len):
        if word_break_in(w, known):                                  # LỖI 2: không bỏ qua ""
            result.append(w)
        known.add(w)
    return result


def wrong_greedy(words: List[str], longest: bool = False) -> List[str]:
    word_set = set(words)
    result = []
    for w in words:
        start, count = 0, 0
        while start < len(w):
            ends = [e for e in range(start + 1, len(w) + 1)
                    if w[start:e] in word_set and not (start == 0 and e == len(w))]
            if not ends:
                break
            start = max(ends) if longest else min(ends)              # LỖI 3: chọn luôn, không quay lui
            count += 1
        if start == len(w) and len(w) > 0:
            result.append(w)
    return result

w1 = examples[0][0]
print("Lỗi 1 (không loại chính nó):", wrong_no_exclude(["cat", "dog", "catdog"]))
print("Lỗi 2 (chuỗi rỗng)         :", wrong_empty(["", "a", "aa"]))
print("Lỗi 3 (tham lam ngắn nhất) :", wrong_greedy(w1))
print("Lỗi 3 (tham lam dài nhất)  :", wrong_greedy(["a", "aa", "ab", "aab"], longest=True))
print("Đúng (Ví dụ 1)             :", concat_dp(w1))
print("Đúng (['a','aa','ab','aab']):", concat_dp(["a", "aa", "ab", "aab"]))

Lỗi 1 (không loại chính nó): ['cat', 'dog', 'catdog']
Lỗi 2 (chuỗi rỗng)         : ['', 'aa']
Lỗi 3 (tham lam ngắn nhất) : ['ratcatdogcat']
Lỗi 3 (tham lam dài nhất)  : ['aa']
Đúng (Ví dụ 1)             : ['catsdogcats', 'dogcatsdog', 'ratcatdogcat']
Đúng (['a','aa','ab','aab']): ['aa', 'aab']


### Lỗi 4: Dùng chung memo **theo vị trí** giữa các từ khác nhau

Memo theo **vị trí** `start` chỉ có nghĩa **trong một từ**: "từ vị trí 1 trở đi của `"ab"`" (là `"b"`, cắt được) khác hẳn "từ vị trí 1 trở đi của `"ac"`" (là `"c"`, không cắt được). Nếu tạo `memo = {}` **một lần** bên ngoài vòng lặp các từ, kết quả của từ trước sẽ bị dùng nhầm cho từ sau.

Ngược lại, Cách 4 dùng memo theo **chuỗi** nên dùng chung giữa các từ là **đúng** — khóa là chính nội dung chuỗi, không phải vị trí.

Sửa: memo theo vị trí phải tạo **mới cho mỗi từ** (hoặc dùng khóa là chuỗi).

In [30]:
def wrong_shared_position_memo(words: List[str]) -> List[str]:
    word_set = set(words)
    memo = {}                                  # LỖI 4: tạo một lần cho MỌI từ

    result = []
    for w in words:
        L = len(w)

        def can(start: int) -> bool:
            # hậu tố w[start:] có cắt được thành >= 1 từ không?
            if start == L:
                return True
            if start in memo:                  # khóa chỉ có start -> lẫn giữa các từ
                return memo[start]
            ans = any(w[start:e] in word_set and can(e) for e in range(start + 1, L + 1))
            memo[start] = ans
            return ans

        # mảnh đầu w[:e] phải ngắn hơn w (e < L) để bảo đảm >= 2 mảnh
        if any(w[:e] in word_set and can(e) for e in range(1, L)):
            result.append(w)
    return result

words = ["ab", "ac", "a", "b"]
print("Sai :", wrong_shared_position_memo(words), " <- 'ac' bị nhận nhầm vì dùng memo[1] của 'ab'")
print("Đúng:", concat_dp(words))

Sai : ['ab', 'ac']  <- 'ac' bị nhận nhầm vì dùng memo[1] của 'ab'
Đúng: ['ab']


### Lỗi 5: Thêm từ vào `known` **trước** khi kiểm tra (Cách 3)

Nếu viết `known.add(w)` rồi mới `word_break_in(w, known)`, ta lại quay về Lỗi 1: `w` tự ghép bằng chính nó. Thứ tự đúng: **kiểm tra trước, thêm sau** (giống Two Sum: "kiểm tra trước khi lưu").

### Lỗi 6: Tra từ trong `list` thay vì `set`

`w[j:i] in words` với `words` là list `10^4` phần tử vẫn cho kết quả đúng, nhưng mỗi lần tra phải dò cả list → chậm gấp hàng nghìn lần, dễ bị **TLE (Time Limit Exceeded — quá thời gian)**. Luôn `word_set = set(words)` trước.

### Lỗi 7: So sánh kết quả không sắp xếp

`assert result == expected` có thể báo sai dù code đúng (Cách 3 trả theo thứ tự độ dài). Dùng `sorted(...)` / `same_answer`.

### Lỗi 8: `lru_cache` đặt lên method có `self`, hoặc đặt ở mức module

Nếu hàm `dfs` được cache ở mức module và đọc `word_set` của **lần gọi trước**, test case sau sẽ dùng nhầm đáp án cũ. Đặt `@lru_cache` lên **hàm con bên trong** method như `SolutionDFS`.

In [31]:
def wrong_add_before_check(words: List[str]) -> List[str]:
    known, result = set(), []
    for w in sorted(words, key=len):
        if not w:
            continue
        known.add(w)                               # LỖI 5: thêm trước
        if word_break_in(w, known):
            result.append(w)
    return result

print("Lỗi 5 (thêm trước khi kiểm tra):", wrong_add_before_check(["cat", "dog", "catdog"]))
print("Đúng                           :", concat_sorted(["cat", "dog", "catdog"]))

Lỗi 5 (thêm trước khi kiểm tra): ['cat', 'dog', 'catdog']
Đúng                           : ['catdog']


## 21. Mẹo nhận dạng pattern

- "Một chuỗi có **cắt được** thành các mảnh thuộc một tập không?" → **Word Break DP**, trạng thái là **một chỉ số** `i` (tiền tố) hoặc `start` (hậu tố).
- "**Với mỗi** phần tử của danh sách, kiểm tra nó có tạo được từ **các phần tử khác**" → chạy bài con cho từng phần tử, và **cẩn thận loại chính nó** (hoặc chỉ dùng phần tử "nhỏ hơn" bằng cách **sắp xếp** rồi thêm dần — pattern giống Longest String Chain, Largest Divisible Subset).
- "Mảnh phải **ngắn hơn**" / "ít nhất 2 phần" → nghĩ tới việc sắp theo độ dài, hoặc cấm mảnh phủ toàn bộ.
- Thấy rất nhiều thao tác "tiền tố của chuỗi có trong tập từ không" → cân nhắc **Trie**.
- Đệ quy thử–sai có **bài con lặp lại** (cùng vị trí / cùng hậu tố) → thêm **memo** hoặc chuyển sang **DP bảng**.
- Chỉ hỏi **có/không** → dùng `break` / `return True` sớm; khác với bài **liệt kê tất cả** (Word Break II) phải đi hết.

## 22. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| 139. Word Break | Bài con của 472: một chuỗi có cắt được không (mục 3) |
| 140. Word Break II | Liệt kê **mọi** cách cắt — đầu ra có thể hàm mũ, cần backtracking + memo |
| 212. Word Search II | Trie + DFS trên lưới để tìm nhiều từ cùng lúc |
| 720. Longest Word in Dictionary | Sắp xếp + `set` "thêm dần", từ được xây từng ký tự từ các từ ngắn hơn |
| 1048. Longest String Chain | Sắp theo độ dài, mỗi từ dựa trên các từ ngắn hơn 1 ký tự |
| 2707. Extra Characters in a String | Word Break DP nhưng tối thiểu số ký tự thừa |
| Biến thể tự chế | Số mảnh ít nhất / số cách ghép / ghép đúng 2 từ (xem Bài tập 5, 6) |

## 23. Bài tập vận dụng

### Bài 1 — Đoán output (không chạy code)

`Solution().findAllConcatenatedWordsInADict` trả về gì (sắp xếp tăng dần) cho mỗi input?

```python
(a) ["a", "b", "ab", "abc"]
(b) ["a", "aa", "aaa", "aaaa"]
(c) ["cat", "catcat", "catcatcat", "dog", "catdogcat"]
(d) ["", "a", "aa"]          # input kiểu phiên bản đề cũ
(e) ["ab", "abab", "ba", "aba"]
```

### Bài 2 — Tính tay bảng `dp`

Với Ví dụ 1, tính bảng `dp` của Cách 2 cho từ `w = "dogcatsdog"` (chỉ số `d0 o1 g2 c3 a4 t5 s6 d7 o8 g9`):

1. Những ô `dp[i]` nào bằng `True`? Với mỗi ô, `j` đầu tiên (nhỏ nhất) thỏa là bao nhiêu?
2. Ô nào `True` nhưng **không** nằm trên cách cắt cuối cùng?
3. Dựng lại cách cắt bằng `parent`.

### Bài 3 — Điền TODO: kiểm tra một từ bằng DP

Hoàn thiện `exercise_is_concatenated(w, word_set)` trả về `True` nếu `w` ghép được từ ≥ 2 từ trong `word_set` (`word_set` **có chứa** `w`). Nhớ hai chi tiết phòng thủ.

In [32]:
def exercise_is_concatenated(w: str, word_set: set) -> bool:
    L = len(w)
    # TODO 1: chuỗi rỗng -> trả False
    dp = [False] * (L + 1)
    # TODO 2: đặt điểm tựa dp[0]
    # TODO 3: hai vòng lặp i (1..L) và j (0..i-1)
    # TODO 4: bỏ qua cặp (j, i) = (0, L)
    # TODO 5: nếu dp[j] và w[j:i] là từ -> dp[i] = True, break
    pass

# Tự làm xong rồi mới mở test:
# ws = set(["cat", "cats", "catsdogcats", "dog", "dogcatsdog", "hippopotamuses", "rat", "ratcatdogcat"])
# assert exercise_is_concatenated("catsdogcats", ws) == True
# assert exercise_is_concatenated("cats", ws) == False
# assert exercise_is_concatenated("", ws | {""}) == False

### Bài 4 — Tự viết lại Cách 3 (sắp theo độ dài)

Không nhìn lên trên, tự viết `exercise_concat_sorted(words)`. Nhớ: sắp theo `len`, bỏ qua chuỗi rỗng, **kiểm tra trước — thêm sau**. Có thể kết hợp giới hạn độ dài mảnh `max_len` (độ dài lớn nhất trong `known`) để nhanh hơn.

In [33]:
def exercise_concat_sorted(words: List[str]) -> List[str]:
    # TODO: known = set(); duyệt sorted(words, key=len); Word Break trên known; known.add(w) sau
    raise NotImplementedError

# Gợi ý test:
# assert same_answer(exercise_concat_sorted(examples[0][0]), examples[0][1])
# assert exercise_concat_sorted(["cat", "dog", "catdog"]) == ["catdog"]
# assert exercise_concat_sorted(["", "a", "aa"]) == ["aa"]
# assert exercise_concat_sorted(["a"]) == []

### Bài 5 — Biến thể: từ ghép từ **đúng hai** từ

Viết `exercise_two_word_concat(words)` trả về mọi từ `w` trong `words` mà `w = x + y` với `x`, `y` là **hai từ** (có thể giống nhau) trong `words`. Yêu cầu: **không dùng DP** — chỉ cần thử mọi điểm cắt, `O(ℓ)` điểm cắt cho mỗi từ.

Ví dụ: `["cat", "dog", "catdog", "catcat", "dogdogcat"]` → `["catdog", "catcat"]` (`"dogdogcat"` cần 3 mảnh `dog + dog + cat` nên **không** tính; cẩn thận: `"catdogcat"` thì **có** tính, vì `"catdog" + "cat"` là 2 từ!). Câu hỏi phụ: kết quả bài này có luôn là tập con của kết quả bài 472 không? Vì sao?

In [34]:
def exercise_two_word_concat(words: List[str]) -> List[str]:
    # TODO: với mỗi w, thử i trong 1..len(w)-1: w[:i] và w[i:] đều là từ?
    pass

# Gợi ý test:
# assert sorted(exercise_two_word_concat(["cat", "dog", "catdog", "catcat", "dogdogcat"])) == ["catcat", "catdog"]
# assert exercise_two_word_concat(examples[0][0]) == []
# assert exercise_two_word_concat(["", "a", "aa"]) == ["aa"]

### Bài 6 — Mini challenge: đếm số cách ghép

Viết `exercise_count_ways(words)` trả về `dict` ánh xạ **mỗi từ ghép** → **số cách** cắt nó thành ≥ 2 từ trong `words` (hai cách khác nhau nếu dãy mảnh khác nhau). Từ không ghép được thì không có trong `dict`.

Ví dụ: `["a", "aa", "aaa"]` → `{"aa": 1, "aaa": 3}` (`"aaa"` = `a+a+a`, `a+aa`, `aa+a`).

Gợi ý suy nghĩ: đếm **mọi** cách cắt bằng DP `ways[i] = Σ ways[j]` với từ điển chứa cả `w`, rồi trừ đi cách "1 mảnh là chính nó". Phép trừ này đúng vì sao?

In [35]:
def exercise_count_ways(words: List[str]) -> dict:
    # TODO: với mỗi w khác rỗng: ways[0] = 1; ways[i] = tổng ways[j] với w[j:i] là từ
    #       số cách >= 2 mảnh = ways[L] - 1 (vì w có trong words)
    pass

# Gợi ý test:
# assert exercise_count_ways(["a", "aa", "aaa"]) == {"aa": 1, "aaa": 3}
# assert exercise_count_ways(examples[0][0]) == {"catsdogcats": 1, "dogcatsdog": 1, "ratcatdogcat": 1}
# assert exercise_count_ways(["a", "aa", "aaa", "aaaa"])["aaaa"] == 7

## 24. Gợi ý theo mức

**Bài 1**
- Mức 1: Với mỗi từ, hỏi: có cách cắt ≥ 2 mảnh nào mà mọi mảnh là từ trong danh sách?
- Mức 2: (b) mọi từ dài ≥ 2 đều cắt được thành các `"a"`. (c) `"catdogcat"` = `cat + dog + cat`.
- Mức 3: (e) `"abab" = "ab" + "ab"`; `"aba" = "ab" + "a"`? — `"a"` có trong danh sách không?

**Bài 2**
- Mức 1: Tìm mọi vị trí bắt đầu/kết thúc của các từ `dog`, `cat`, `cats` trong `w`.
- Mức 2: `dp[3]` nhờ `"dog"`; từ `j = 3` có hai từ bắt đầu: `"cat"` và `"cats"`.
- Mức 3: Chạy `trace_concat_dp("dogcatsdog", ws)` để đối chiếu (sau khi tự tính!).

**Bài 3**
- Mức 1: Giống hệt `word_break_139`, thêm 2 dòng.
- Mức 2: `if L == 0: return False` và `if j == 0 and i == L: continue`.
- Mức 3: Điều kiện bật: `if dp[j] and w[j:i] in word_set: dp[i] = True; break`; cuối cùng `return dp[L]`.

**Bài 4**
- Mức 1: `for w in sorted(words, key=len)`.
- Mức 2: `if not w: continue`, rồi Word Break của `w` trên `known`, rồi `known.add(w)`.
- Mức 3: Word Break: `dp[0] = True`; `dp[i] |= dp[j] and w[j:i] in known`.

**Bài 5**
- Mức 1: `for i in range(1, len(w))` — cả hai nửa khác rỗng.
- Mức 2: `if w[:i] in word_set and w[i:] in word_set`.
- Mức 3: Tập con: 2 mảnh ≥ 2 mảnh, và mỗi nửa ngắn hơn `w` → mọi kết quả Bài 5 là từ ghép của bài 472.

**Bài 6**
- Mức 1: Thay "có/không" (`True/False`) bằng "bao nhiêu" (`int`), thay `or` bằng `+`, bỏ `break`.
- Mức 2: `ways[i] += ways[j]` khi `w[j:i] in word_set`.
- Mức 3: Trong `ways[L]` có đúng **một** cách dùng 1 mảnh (chính `w`, vì `w` có trong `words` và mọi mảnh khác `w` thì ngắn hơn) → trừ 1. Chỉ đưa vào `dict` khi kết quả > 0.

## 25. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

- (a) `["ab"]` — `"abc"` không cắt được vì không có từ `"c"`, `"bc"`.
- (b) `["aa", "aaa", "aaaa"]` — chỉ `"a"` không ghép được (chỉ có 1 ký tự).
- (c) `["catcat", "catcatcat", "catdogcat"]` — mảnh dùng lại được.
- (d) `["aa"]` — chuỗi rỗng không được tính, và cũng không làm `"a"` thành `"" + "a"`.
- (e) `["abab"]` — `"abab" = "ab" + "ab"`; `"aba"` cần `"ab" + "a"` hoặc `"a" + "ba"` nhưng không có từ `"a"` → không ghép được. Người mới hay vội trả lời có `"aba"`.

### Đáp án Bài 2

| i | `w[:i]` | `j` đầu tiên thỏa | Mảnh | `dp[i]` |
|---:|---|---:|---|:---:|
| 0 | `""` | — | điểm tựa | T |
| 3 | `"dog"` | 0 | `"dog"` | T |
| 6 | `"dogcat"` | 3 | `"cat"` | T |
| 7 | `"dogcats"` | 3 | `"cats"` | T |
| 10 | `"dogcatsdog"` | 7 | `"dog"` (`j=0` bị cấm; `j=3` thử `"catsdog"`, `j=6` thử `"sdog"` không được) | T |

Ô `dp[6]` bật nhưng không nằm trên cách cắt cuối (sau `"dogcat"` còn `"sdog"` — không phải từ). Lần ngược `parent`: `10 ← 7 ← 3 ← 0` → `"dog" + "cats" + "dog"`.

In [36]:
# Kiểm chứng Bài 1
quiz = {
    "a": ["a", "b", "ab", "abc"],
    "b": ["a", "aa", "aaa", "aaaa"],
    "c": ["cat", "catcat", "catcatcat", "dog", "catdogcat"],
    "d": ["", "a", "aa"],
    "e": ["ab", "abab", "ba", "aba"],
}
answers = {
    "a": ["ab"],
    "b": ["aa", "aaa", "aaaa"],
    "c": ["catcat", "catcatcat", "catdogcat"],
    "d": ["aa"],
    "e": ["abab"],
}
for key, words in quiz.items():
    got = sorted(Solution().findAllConcatenatedWordsInADict(words))
    print(f"({key}) {got}")
    assert got == answers[key]

# Kiểm chứng Bài 2
print()
ws = set(examples[0][0])
assert trace_concat_dp("dogcatsdog", ws) is True
print("Bài 1, Bài 2 PASS")

(a) ['ab']
(b) ['aa', 'aaa', 'aaaa']
(c) ['catcat', 'catcatcat', 'catdogcat']
(d) ['aa']
(e) ['abab']

  dp[ 3] = True  (j=0, mảnh 'dog')
  dp[ 6] = True  (j=3, mảnh 'cat')
  dp[ 7] = True  (j=3, mảnh 'cats')
  dp[10] = True  (j=7, mảnh 'dog')
  chỉ số : 01234567890
  bảng dp: T..T..TT..T -> True
  cách cắt: dog + cats + dog
Bài 1, Bài 2 PASS


In [37]:
# ===== Đáp án Bài 3 và Bài 4 =====
def answer_is_concatenated(w: str, word_set: set) -> bool:
    L = len(w)
    if L == 0:
        return False
    dp = [False] * (L + 1)
    dp[0] = True
    for i in range(1, L + 1):
        for j in range(i):
            if j == 0 and i == L:
                continue
            if dp[j] and w[j:i] in word_set:
                dp[i] = True
                break
    return dp[L]


def answer_concat_sorted(words: List[str]) -> List[str]:
    known, result = set(), []
    max_len = 0                                   # độ dài lớn nhất trong known
    for w in sorted(words, key=len):
        if not w:
            continue
        L = len(w)
        dp = [False] * (L + 1)
        dp[0] = True
        for i in range(1, L + 1):
            for j in range(max(0, i - max_len), i):   # mảnh dài hơn max_len chắc chắn không có
                if dp[j] and w[j:i] in known:
                    dp[i] = True
                    break
        if dp[L]:
            result.append(w)
        known.add(w)
        max_len = L                               # sắp tăng dần nên L luôn là lớn nhất
    return result


ws = set(examples[0][0])
assert answer_is_concatenated("catsdogcats", ws) == True
assert answer_is_concatenated("cats", ws) == False
assert answer_is_concatenated("", ws | {""}) == False
assert same_answer(answer_concat_sorted(examples[0][0]), examples[0][1])
assert answer_concat_sorted(["cat", "dog", "catdog"]) == ["catdog"]
assert answer_concat_sorted(["", "a", "aa"]) == ["aa"]
assert answer_concat_sorted(["a"]) == []

random.seed(4)
for _ in range(300):
    words = list({"".join(random.choice("ab") for _ in range(random.randint(1, 7)))
                  for _ in range(random.randint(1, 10))})
    assert sorted(answer_concat_sorted(words)) == sorted(concat_brute(words))
    assert [w for w in words if answer_is_concatenated(w, set(words))] == concat_dp(words)
print("Bài 3, Bài 4 PASS (kèm 300 test đối chiếu)")

Bài 3, Bài 4 PASS (kèm 300 test đối chiếu)


In [38]:
# ===== Đáp án Bài 5 và Bài 6 =====
def answer_two_word_concat(words: List[str]) -> List[str]:
    word_set = set(words)
    result = []
    for w in words:
        for i in range(1, len(w)):                # hai nửa đều khác rỗng
            if w[:i] in word_set and w[i:] in word_set:
                result.append(w)
                break
    return result


def answer_count_ways(words: List[str]) -> dict:
    word_set = set(w for w in words if w)
    result = {}
    for w in words:
        L = len(w)
        if L == 0:
            continue
        ways = [0] * (L + 1)
        ways[0] = 1                               # tiền tố rỗng: 1 cách (0 mảnh)
        for i in range(1, L + 1):
            for j in range(i):
                if ways[j] and w[j:i] in word_set:
                    ways[i] += ways[j]
        count = ways[L] - 1                       # trừ cách "1 mảnh = chính w"
        if count > 0:
            result[w] = count
    return result


assert sorted(answer_two_word_concat(["cat", "dog", "catdog", "catcat", "dogdogcat"])) == ["catcat", "catdog"]
assert answer_two_word_concat(examples[0][0]) == []
assert answer_two_word_concat(["", "a", "aa"]) == ["aa"]

assert answer_count_ways(["a", "aa", "aaa"]) == {"aa": 1, "aaa": 3}
assert answer_count_ways(examples[0][0]) == {"catsdogcats": 1, "dogcatsdog": 1, "ratcatdogcat": 1}
assert answer_count_ways(["a", "aa", "aaa", "aaaa"])["aaaa"] == 7
print("Ví dụ đếm cách:", answer_count_ways(["a", "aa", "aaa", "aaaa"]))

# Đối chiếu: Bài 5 là tập con của 472; khóa của Bài 6 đúng bằng tập từ ghép của 472
random.seed(6)
for _ in range(300):
    words = list({"".join(random.choice("ab") for _ in range(random.randint(1, 7)))
                  for _ in range(random.randint(1, 10))})
    full = set(concat_brute(words))
    assert set(answer_two_word_concat(words)) <= full
    assert set(answer_count_ways(words)) == full
print("Bài 5, Bài 6 PASS (kèm 300 test đối chiếu)")

Ví dụ đếm cách: {'aa': 1, 'aaa': 3, 'aaaa': 7}
Bài 5, Bài 6 PASS (kèm 300 test đối chiếu)


## 26. Template ngắn nên nhớ

```python
class Solution:
    def findAllConcatenatedWordsInADict(self, words: List[str]) -> List[str]:
        word_set = set(words)
        result = []
        for w in words:
            L = len(w)
            if L == 0:                       # chuỗi rỗng: không bao giờ là từ ghép
                continue
            dp = [False] * (L + 1)
            dp[0] = True                     # tiền tố rỗng
            for i in range(1, L + 1):
                for j in range(i):
                    if j == 0 and i == L:    # cấm 1 mảnh = chính w
                        continue
                    if dp[j] and w[j:i] in word_set:
                        dp[i] = True
                        break
            if dp[L]:
                result.append(w)
        return result
```

Câu thần chú: **"Mỗi từ chạy một Word Break — trên các từ khác nó — và không bao giờ nhận chuỗi rỗng."**

Bản 5 dòng để nhớ ý tưởng DFS: `dfs(s) = any(s[:i] in S and (s[i:] in S or dfs(s[i:])) for i in 1..len(s)-1)`.

## 27. Checklist tự kiểm tra trước khi submit

- Đã đổi `words` sang `set` để tra `O(1)` trung bình chưa?
- Đã **loại chính từ đang xét** (cấm cặp `(0, L)`, `discard`/`add`, hoặc sắp theo độ dài và **kiểm tra trước — thêm sau**) chưa?
- Đã **bỏ qua chuỗi rỗng** (không kiểm tra nó, không dùng nó làm mảnh) chưa?
- Có lỡ dùng **tham lam** (lấy mảnh đầu tiên rồi không quay lại) không?
- Memo theo **vị trí** có được tạo **mới cho mỗi từ** không? (Memo theo **chuỗi** thì dùng chung được.)
- `lru_cache` có đặt trên **hàm con bên trong** method không?
- Đã test: 2 ví dụ đề, danh sách 1 từ, mảnh dùng lại (`"catcat"`), chuỗi rỗng, từ bẫy `"a"*29 + "b"`, ca cần quay lui (`"catsdogcats"`)?
- Khi tự test đã so sánh **bỏ qua thứ tự** (`sorted`) chưa?